In [ ]:
# Safe Jupyter startup on Linux only
import os, sys

if sys.platform.startswith("linux"):
    os.environ.setdefault("MPLBACKEND", "Agg")        # avoids Qt/OpenGL issues
    os.environ.setdefault("QT_QPA_PLATFORM", "offscreen")
    #os.environ.setdefault("OMP_NUM_THREADS", "1")     # tames OpenMP storms
    os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
    #os.environ.setdefault("CUDA_VISIBLE_DEVICES", "") # comment out if you WANT GPU
else:
    # macOS/Windows: leave defaults; optionally keep OMP cap
    #os.environ.setdefault("OMP_NUM_THREADS", "1")
    pass


In [ ]:
# Import necessary libraries

import numpy as np
from pathlib import Path
import sys
from datetime import datetime
import matplotlib.pyplot as plt
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import gc
import io
import json

from scipy.stats import linregress
from scipy.signal import butter, filtfilt
from scipy.signal import correlate
from scipy.stats import pearsonr

from harp_resources import process
import aeon.io.api as api
from sleap import load_and_process as lp

# Reload modules to pick up latest changes (useful after code updates)
# Set force_reload_modules = True to always reload, or False to use cached
# versions
# Set to False for faster execution when modules haven't changed
force_reload_modules = False
if force_reload_modules:
    import importlib
    import sleap.load_and_process
    import sleap.processing_functions
    import sleap.visualization

    importlib.reload(sleap.load_and_process)
    importlib.reload(sleap.processing_functions)
    importlib.reload(sleap.visualization)
    # Re-import aliases after reload
    lp = sleap.load_and_process
 #   pf = sleap.processing_functions


def get_eye_label(key):
    """Return mapped user-viewable eye label for video key."""
    return VIDEO_LABELS.get(key, key)


# Column prefixes that indicate SLEAP-derived eye-tracking data
_SLEAP_EYE_PREFIXES = (
    "left",
    "right",
    "center",
    "p1",
    "p2",
    "p3",
    "p4",
    "p5",
    "p6",
    "p7",
    "p8",
    "Ellipse",
)

RESAMPLED_DROP_COLUMNS = [
    "Ellipse.Angle",
    "Ellipse.Diameter",
    "instance.score",
    "left.x",
    "left.y",
    "left.score",
    "right.x",
    "right.y",
    "right.score",
    "p1.x",
    "p1.y",
    "p1.score",
    "p2.x",
    "p2.y",
    "p2.score",
    "p3.x",
    "p3.y",
    "p3.score",
    "p4.x",
    "p4.y",
    "p4.score",
    "p5.x",
    "p5.y",
    "p5.score",
    "p6.x",
    "p6.y",
    "p6.score",
    "p7.x",
    "p7.y",
    "p7.score",
    "p8.x",
    "p8.y",
    "p8.score",
    "center.x",
    "center.y",
    "center.score",
]


def _needs_eye_suffix(column: str) -> bool:
    """Return True if the column should be tagged as eye data during resampling."""
    return any(column.startswith(prefix) for prefix in _SLEAP_EYE_PREFIXES)


def resample_video_dataframe(
    video_df: pd.DataFrame,
    eye_tag: str,
    target_freq_hz: float = None,
    optical_filter_hz: float = None,
) -> pd.DataFrame:
    """Resample a SLEAP video dataframe onto the common time grid."""
    if "Seconds" not in video_df.columns:
        raise ValueError("Video dataframe must contain a 'Seconds' column before resampling.")

    target_freq_hz = target_freq_hz or COMMON_RESAMPLED_RATE
    df_for_resample = video_df.copy()

    # Rename SLEAP-specific columns so the resampler treats them as eye data
    rename_map = {
        col: f"{col}_{eye_tag}"
        for col in df_for_resample.columns
        if col not in {"Seconds", "frame_idx"} and _needs_eye_suffix(col)
    }
    df_for_resample = df_for_resample.rename(columns=rename_map)

    # Convert Seconds to aeon datetime index and drop Seconds before resampling
    df_for_resample.index = pd.to_datetime(df_for_resample["Seconds"].apply(api.aeon))
    df_for_resample = df_for_resample.drop(columns=["Seconds"])

    resampled = process.resample_dataframe(
        df_for_resample,
        target_freq_Hz=target_freq_hz,
        optical_filter_Hz=optical_filter_hz,
    )

    # Restore original column names
    inverse_map = {v: k for k, v in rename_map.items()}
    resampled = resampled.rename(columns=inverse_map)

    # Convert index back to Seconds
    resampled_seconds = (resampled.index - datetime(1904, 1, 1)).total_seconds()
    resampled = resampled.reset_index(drop=True)
    resampled.insert(0, "Seconds", resampled_seconds)

    # Frame indices should remain integers if present
    if "frame_idx" in resampled.columns:
        resampled["frame_idx"] = (
            pd.to_numeric(resampled["frame_idx"], errors="coerce").round().astype("Int64")
        )

    return resampled


def set_aeon_index(df: pd.DataFrame) -> pd.DataFrame:
    """Return a copy of the dataframe with a DatetimeIndex derived from Seconds."""
    df_indexed = df.copy()
    df_indexed.index = pd.to_datetime(df_indexed["Seconds"].apply(api.aeon))
    df_indexed.index.name = "aeon_time"
    return df_indexed


def append_aeon_time_column(df: pd.DataFrame) -> pd.DataFrame:
    """Return a copy of the dataframe with an additional aeon_time ISO timestamp column."""
    df_with_time = df.copy()
    if isinstance(df_with_time.index, pd.DatetimeIndex):
        df_with_time["aeon_time"] = df_with_time.index.map(lambda ts: ts.isoformat())
    else:
        df_with_time["aeon_time"] = df_with_time["Seconds"].apply(lambda x: api.aeon(x).isoformat())
    return df_with_time


def compute_blink_instance_score_threshold(
    video_df: pd.DataFrame,
    hard_threshold: float,
    use_adaptive: bool = True,
    adaptive_percentile: float = 5.0,
    min_threshold: float = 2.5,
    max_threshold: float = 4.5,
    score_column: str = "instance.score",
    adaptive_method: str = "histogram_valley",
    allow_upward: bool = False,
    min_peak_separation: float = 1.0,
    min_low_fraction: float = 0.0005,
    max_low_fraction: float = 0.2,
) -> float:
    """Return blink threshold from score distribution with robust fallbacks.

    Default behavior is adaptive-down-only: adaptive threshold can lower, but not
    increase, the trusted hard threshold.
    """
    hard_threshold = float(hard_threshold)
    if not use_adaptive:
        return hard_threshold

    if score_column not in video_df.columns:
        return hard_threshold

    scores = pd.to_numeric(video_df[score_column], errors="coerce").dropna().to_numpy()
    if scores.size < 100:
        return hard_threshold

    candidate = None

    # Distribution-shape method: find valley between low-score and high-score modes.
    if adaptive_method == "histogram_valley":
        hist_min = float(np.nanmin(scores))
        hist_max = float(np.nanmax(scores))
        if np.isfinite(hist_min) and np.isfinite(hist_max) and hist_max > hist_min:
            edges = np.linspace(hist_min, hist_max, 121)
            hist, _ = np.histogram(scores, bins=edges)
            centers = (edges[:-1] + edges[1:]) / 2
            # Smooth histogram to reduce spurious local extrema.
            kernel = np.array([1, 2, 3, 2, 1], dtype=float)
            hist_smooth = np.convolve(hist.astype(float), kernel / kernel.sum(), mode="same")

            left_mask = centers < hard_threshold
            right_mask = centers >= hard_threshold

            if left_mask.any() and right_mask.any():
                left_idxs = np.where(left_mask)[0]
                right_idxs = np.where(right_mask)[0]
                left_peak = left_idxs[np.argmax(hist_smooth[left_mask])]
                right_peak = right_idxs[np.argmax(hist_smooth[right_mask])]

                peak_sep = abs(float(centers[right_peak] - centers[left_peak]))
                if peak_sep >= min_peak_separation:
                    lo, hi = sorted((left_peak, right_peak))
                    valley = lo + int(np.argmin(hist_smooth[lo : hi + 1]))
                    valley_threshold = float(centers[valley])

                    # Quality gate: low-score component should be plausible in size.
                    low_fraction = float(np.mean(scores <= valley_threshold))
                    if min_low_fraction <= low_fraction <= max_low_fraction:
                        candidate = valley_threshold

    # Fallback adaptive candidate if valley method is unavailable/unreliable.
    if candidate is None:
        candidate = float(np.nanpercentile(scores, adaptive_percentile))

    candidate = float(np.clip(candidate, min_threshold, max_threshold))

    # Adaptive-down-only mode: never exceed trusted hard threshold.
    if not allow_upward:
        return min(hard_threshold, candidate)
    return candidate

def add_blink_flag_column(video_df: pd.DataFrame, blink_segments: list) -> pd.DataFrame:
    """Return a copy of the dataframe with binary blink_flag (1=in blink, 0=not blink)."""
    df = video_df.copy()
    df["blink_flag"] = 0

    if not blink_segments:
        return df

    n_rows = len(df)
    if n_rows == 0:
        return df

    for seg in blink_segments:
        if not isinstance(seg, dict):
            continue
        if "start_idx" not in seg or "end_idx" not in seg:
            continue
        start_idx = max(0, int(seg["start_idx"]))
        end_idx = min(n_rows - 1, int(seg["end_idx"]))
        if end_idx >= start_idx:
            df.loc[start_idx:end_idx, "blink_flag"] = 1

    # Keep nullable integer type so parquet reads cleanly in pandas
    df["blink_flag"] = pd.to_numeric(df["blink_flag"], errors="coerce").fillna(0).astype("Int64")
    return df

# symbols to use ✅ ℹ️ ⚠️ ❗

In [ ]:
# set up variables and load data
##########################################################################

data_path = Path(
    "/Users/rancze/Documents/Data/vestVR/20250409_Cohort3_rotation/Visual_mismatch_day4/B6J2783-2025-04-28T14-57-30"
)
# data_path =
# Path('/Users/rancze/Documents/Data/vestVR/Cohort1/No_iso_correction/Visual_mismatch_day3/B6J2717-2024-12-10T12-17-03') only has sleap data 1 for testing purposes

### MOST commonly changed params for tuning
video1_eye = "L" # Options: 'L' or 'R'; which eye does VideoData1 represent? ('L' = Left,'R' = Right)
debug = True # Set to True to enable verbose debug output and plots across all cells (file loading, processing, etc.)
plot_QC = True # plot QC plots
plot_QC_timeseries = False # plot timeseries (i.e. whole pupil position) - quite heavy
blink_instance_score_threshold = 3.8 # base hard threshold for blink detection fallback
use_adaptive_blink_threshold = True # if True, compute per-video adaptive threshold from instance.score
adaptive_blink_method = "histogram_valley" # "histogram_valley" (default) or percentile fallback behavior
adaptive_blink_allow_upward = False # False = adaptive-down-only (never above hard threshold)
adaptive_blink_percentile = 5.0 # fallback percentile if valley method is unreliable
adaptive_blink_min_peak_separation = 1.0 # minimum separation between low/high histogram peaks
adaptive_blink_min_low_fraction = 0.0005 # quality gate: minimum fraction below candidate threshold
adaptive_blink_max_low_fraction = 0.2 # quality gate: maximum fraction below candidate threshold
blink_threshold_min = 2.5 # lower clamp for adaptive threshold
blink_threshold_max = 4.5 # upper clamp for adaptive threshold

### SLEAP raw data filtering 
score_cutoff = 0.2 # for filtering out inferred points with low confidence inference scores, they get interpolated
max_low_confidence_interpolation_fraction = 0.05 # if low-confidence coordinate values exceed this fraction, exclude that VideoData from downstream processing
outlier_sd_threshold = 10  # for removing outliers from the data after low-confidence instance values are filtered out, they get interpolated

pupil_filter_cutoff_hz = 10  # Hz, Pupil diameter filter settings (Butterworth low-pass)
pupil_filter_order = 6

### Parameters for blink detection
min_blink_duration_ms = 50  # minimum blink duration in milliseconds
blink_merge_window_ms = 100 # NOT CURRENTLY USED: merge window was removed to preserve good data between separate blinks
long_blink_warning_ms = 2000 # warn if blinks exceed this duration (in ms) - user should verify these are real blinks

COMMON_RESAMPLED_RATE = 1000 # Common resampling rate (Hz) used for alignment with other modalities, like photometry, HARP streams, etc. 

NaNs_removed = False #keep as false here, it is to checking if NaNs already removed if the notebook cell is rerun

# Automatically assign eye for VideoData2
video2_eye = "R" if video1_eye == "L" else "L"
eye_fullname = {"L": "Left", "R": "Right"} # Map for full names (used in labels)
VIDEO_LABELS = { # Update VIDEO_LABELS based on selection
    "VideoData1": f"VideoData1 ({video1_eye}: {eye_fullname[video1_eye]})",
    "VideoData2": f"VideoData2 ({video2_eye}: {eye_fullname[video2_eye]})",
}

save_path = data_path.parent / f"{data_path.name}_processedData"
qc_debug_dir = save_path / "QC_and_debug"
qc_debug_dir.mkdir(parents=True, exist_ok=True)

VideoData1, VideoData2, VideoData1_Has_Sleap, VideoData2_Has_Sleap = (
    lp.load_videography_data(data_path, debug=debug)
)

# Load manual blink data if available
manual_blinks_v1 = lp.load_manual_blinks(data_path, video_number=1)
manual_blinks_v2 = lp.load_manual_blinks(data_path, video_number=2)

columns_of_interest = [
    "left.x",
    "left.y",
    "center.x",
    "center.y",
    "right.x",
    "right.y",
    "p1.x",
    "p1.y",
    "p2.x",
    "p2.y",
    "p3.x",
    "p3.y",
    "p4.x",
    "p4.y",
    "p5.x",
    "p5.y",
    "p6.x",
    "p6.y",
    "p7.x",
    "p7.y",
    "p8.x",
    "p8.y",
]

if VideoData1_Has_Sleap:
    # drop the track column as it is empty
    if "track" in VideoData1.columns:
        VideoData1 = VideoData1.drop(columns=["track"])
    coordinates_dict1_raw = lp.get_coordinates_dict(VideoData1, columns_of_interest)
    # frame rate for VideoData1 TODO where to save it, is it useful?
    FPS_1 = 1 / VideoData1["Seconds"].diff().mean()
    print()
    print(f"{get_eye_label('VideoData1')}: FPS = {FPS_1}")

if VideoData2_Has_Sleap:
    # drop the track column as it is empty
    if "track" in VideoData2.columns:
        VideoData2 = VideoData2.drop(columns=["track"])
    coordinates_dict2_raw = lp.get_coordinates_dict(VideoData2, columns_of_interest)
    # frame rate for VideoData2
    FPS_2 = 1 / VideoData2["Seconds"].diff().mean()
    print(f"{get_eye_label('VideoData2')}: FPS = {FPS_2}")

In [ ]:
# if needed, plot timeseries of coordinates in browser for both VideoData1 and VideoData2
#########################################################################
if plot_QC_timeseries:
    print(
        "⚠️ Check for long discontinuities and outliers in the data, we will try to deal with them later"
    )
    print("ℹ️ Figures open in browser window, takes a bit of time.")

    # Helper list variables
    subplot_titles = (
        "X coordinates for pupil centre and left-right eye corner",
        "Y coordinates for pupil centre and left-right eye corner",
        "X coordinates for iris points",
        "Y coordinates for iris points",
    )
    eye_x = ["left.x", "center.x", "right.x"]
    eye_y = ["left.y", "center.y", "right.y"]
    iris_x = ["p1.x", "p2.x", "p3.x", "p4.x", "p5.x", "p6.x", "p7.x", "p8.x"]
    iris_y = ["p1.y", "p2.y", "p3.y", "p4.y", "p5.y", "p6.y", "p7.y", "p8.y"]

    # --- VideoData1 ---
    if VideoData1_Has_Sleap:
        fig1 = make_subplots(
            rows=4,
            cols=1,
            shared_xaxes=True,
            vertical_spacing=0.05,
            subplot_titles=subplot_titles,
        )

        # Row 1: left.x, center.x, right.x
        for col in eye_x:
            fig1.add_trace(
                go.Scatter(
                    x=VideoData1["Seconds"], y=VideoData1[col], mode="lines", name=col
                ),
                row=1,
                col=1,
            )
        # Row 2: left.y, center.y, right.y
        for col in eye_y:
            fig1.add_trace(
                go.Scatter(
                    x=VideoData1["Seconds"], y=VideoData1[col], mode="lines", name=col
                ),
                row=2,
                col=1,
            )
        # Row 3: p1.x ... p8.x
        for col in iris_x:
            fig1.add_trace(
                go.Scatter(
                    x=VideoData1["Seconds"], y=VideoData1[col], mode="lines", name=col
                ),
                row=3,
                col=1,
            )
        # Row 4: p1.y ... p8.y
        for col in iris_y:
            fig1.add_trace(
                go.Scatter(
                    x=VideoData1["Seconds"], y=VideoData1[col], mode="lines", name=col
                ),
                row=4,
                col=1,
            )

        fig1.update_layout(
            height=1200,
            title_text=f"Time series subplots for coordinates [{get_eye_label('VideoData1')}]",
            showlegend=True,
        )
        fig1.update_xaxes(title_text="Seconds", row=4, col=1)
        fig1.update_yaxes(title_text="X Position", row=1, col=1)
        fig1.update_yaxes(title_text="Y Position", row=2, col=1)
        fig1.update_yaxes(title_text="X Position", row=3, col=1)
        fig1.update_yaxes(title_text="Y Position", row=4, col=1)

        fig1.show(renderer="browser")

    # --- VideoData2 ---
    if VideoData2_Has_Sleap:
        fig2 = make_subplots(
            rows=4,
            cols=1,
            shared_xaxes=True,
            vertical_spacing=0.05,
            subplot_titles=subplot_titles,
        )
        # Row 1: left.x, center.x, right.x
        for col in eye_x:
            fig2.add_trace(
                go.Scatter(
                    x=VideoData2["Seconds"], y=VideoData2[col], mode="lines", name=col
                ),
                row=1,
                col=1,
            )
        # Row 2: left.y, center.y, right.y
        for col in eye_y:
            fig2.add_trace(
                go.Scatter(
                    x=VideoData2["Seconds"], y=VideoData2[col], mode="lines", name=col
                ),
                row=2,
                col=1,
            )
        # Row 3: p1.x ... p8.x
        for col in iris_x:
            fig2.add_trace(
                go.Scatter(
                    x=VideoData2["Seconds"], y=VideoData2[col], mode="lines", name=col
                ),
                row=3,
                col=1,
            )
        # Row 4: p1.y ... p8.y
        for col in iris_y:
            fig2.add_trace(
                go.Scatter(
                    x=VideoData2["Seconds"], y=VideoData2[col], mode="lines", name=col
                ),
                row=4,
                col=1,
            )

        fig2.update_layout(
            height=1200,
            title_text=f"Time series subplots for coordinates [{get_eye_label('VideoData2')}]",
            showlegend=True,
        )
        fig2.update_xaxes(title_text="Seconds", row=4, col=1)
        fig2.update_yaxes(title_text="X Position", row=1, col=1)
        fig2.update_yaxes(title_text="Y Position", row=2, col=1)
        fig2.update_yaxes(title_text="X Position", row=3, col=1)
        fig2.update_yaxes(title_text="Y Position", row=4, col=1)

        fig2.show(renderer="browser")

In [ ]:
# QC plot XY coordinate distributions to visualize outliers
##########################################################################
columns_of_interest = [
    "left",
    "right",
    "center",
    "p1",
    "p2",
    "p3",
    "p4",
    "p5",
    "p6",
    "p7",
    "p8",
]

# Filter out NaN values and calculate the min and max values for X and Y
# coordinates for both dict1 and dict2

def min_max_dict(coordinates_dict):
    x_min = min(
        [
            coordinates_dict[f"{col}.x"][
                ~np.isnan(coordinates_dict[f"{col}.x"])
            ].min()
            for col in columns_of_interest
        ]
    )
    x_max = max(
        [
            coordinates_dict[f"{col}.x"][
                ~np.isnan(coordinates_dict[f"{col}.x"])
            ].max()
            for col in columns_of_interest
        ]
    )
    y_min = min(
        [
            coordinates_dict[f"{col}.y"][
                ~np.isnan(coordinates_dict[f"{col}.y"])
            ].min()
            for col in columns_of_interest
        ]
    )
    y_max = max(
        [
            coordinates_dict[f"{col}.y"][
                ~np.isnan(coordinates_dict[f"{col}.y"])
            ].max()
            for col in columns_of_interest
        ]
    )
    return x_min, x_max, y_min, y_max

# PLOT QC plot XY coordinate distributions to visualize outliers
##########################################################################
if plot_QC:
    # Compute min/max as before for global axes limits
    if VideoData1_Has_Sleap:
        x_min1, x_max1, y_min1, y_max1 = lp.min_max_dict(
            coordinates_dict1_raw, columns_of_interest
        )

    if VideoData2_Has_Sleap:
        x_min2, x_max2, y_min2, y_max2 = lp.min_max_dict(
            coordinates_dict2_raw, columns_of_interest
        )

    # Use global min and max for consistency only if both VideoData1_Has_Sleap
    # and VideoData2_Has_Sleap are True
    if VideoData1_Has_Sleap and VideoData2_Has_Sleap:
        x_min = min(x_min1, x_min2)
        x_max = max(x_max1, x_max2)
        y_min = min(y_min1, y_min2)
        y_max = max(y_max1, y_max2)
    elif VideoData1_Has_Sleap:
        x_min, x_max, y_min, y_max = x_min1, x_max1, y_min1, y_max1
    elif VideoData2_Has_Sleap:
        x_min, x_max, y_min, y_max = x_min2, x_max2, y_min2, y_max2
    else:
        raise ValueError("Neither VideoData1 nor VideoData2 has Sleap data available.")

    # Create the figure and axes

    fig, ax = plt.subplots(nrows=2, ncols=2, figsize=(18, 12))

    fig.suptitle(
        f"XY coordinate distribution of different points for {get_eye_label('VideoData1')} and {get_eye_label('VideoData2')} before outlier removal and NaN interpolation",
        fontsize=14,
    )

    # Define colormap for p1-p8

    colors = ["blue", "green", "red", "cyan", "magenta", "yellow", "black", "orange"]

    # Panel 1: left, right, center (dict1)

    if "VideoData1_Has_Sleap" in globals() and VideoData1_Has_Sleap:
        ax[0, 0].set_title(f"{get_eye_label('VideoData1')}: left, right, center")
        ax[0, 0].scatter(
            coordinates_dict1_raw["left.x"],
            coordinates_dict1_raw["left.y"],
            color="black",
            label="left",
            s=10,
        )
        ax[0, 0].scatter(
            coordinates_dict1_raw["right.x"],
            coordinates_dict1_raw["right.y"],
            color="grey",
            label="right",
            s=10,
        )
        ax[0, 0].scatter(
            coordinates_dict1_raw["center.x"],
            coordinates_dict1_raw["center.y"],
            color="red",
            label="center",
            s=10,
        )
        ax[0, 0].set_xlim([x_min, x_max])
        ax[0, 0].set_ylim([y_min, y_max])
        ax[0, 0].set_xlabel("x coordinates (pixels)")
        ax[0, 0].set_ylabel("y coordinates (pixels)")
        ax[0, 0].legend(loc="upper right")
    else:
        ax[0, 0].axis("off")

    # Panel 2: p1 to p8 (dict1)

    if "VideoData1_Has_Sleap" in globals() and VideoData1_Has_Sleap:
        ax[0, 1].set_title(f"{get_eye_label('VideoData1')}: p1 to p8")
        for idx, col in enumerate(columns_of_interest[3:]):
            ax[0, 1].scatter(
                coordinates_dict1_raw[f"{col}.x"],
                coordinates_dict1_raw[f"{col}.y"],
                color=colors[idx],
                label=col,
                s=5,
            )

        ax[0, 1].set_xlim([x_min, x_max])
        ax[0, 1].set_ylim([y_min, y_max])
        ax[0, 1].set_xlabel("x coordinates (pixels)")
        ax[0, 1].set_ylabel("y coordinates (pixels)")
        ax[0, 1].legend(loc="upper right")
    else:
        ax[0, 1].axis("off")

    # Panel 3: left, right, center (dict2)

    if "VideoData2_Has_Sleap" in globals() and VideoData2_Has_Sleap:
        ax[1, 0].set_title(f"{get_eye_label('VideoData2')}: left, right, center")
        ax[1, 0].scatter(
            coordinates_dict2_raw["left.x"],
            coordinates_dict2_raw["left.y"],
            color="black",
            label="left",
            s=10,
        )
        ax[1, 0].scatter(
            coordinates_dict2_raw["right.x"],
            coordinates_dict2_raw["right.y"],
            color="grey",
            label="right",
            s=10,
        )
        ax[1, 0].scatter(
            coordinates_dict2_raw["center.x"],
            coordinates_dict2_raw["center.y"],
            color="red",
            label="center",
            s=10,
        )
        ax[1, 0].set_xlim([x_min, x_max])
        ax[1, 0].set_ylim([y_min, y_max])
        ax[1, 0].set_xlabel("x coordinates (pixels)")
        ax[1, 0].set_ylabel("y coordinates (pixels)")
        ax[1, 0].legend(loc="upper right")
    else:
        ax[1, 0].axis("off")

    # Panel 4: p1 to p8 (dict2)

    if "VideoData2_Has_Sleap" in globals() and VideoData2_Has_Sleap:
        ax[1, 1].set_title(f"{get_eye_label('VideoData2')}: p1 to p8")
        for idx, col in enumerate(columns_of_interest[3:]):
            ax[1, 1].scatter(
                coordinates_dict2_raw[f"{col}.x"],
                coordinates_dict2_raw[f"{col}.y"],
                color=colors[idx],
                label=col,
                s=5,
            )

        ax[1, 1].set_xlim([x_min, x_max])
        ax[1, 1].set_ylim([y_min, y_max])
        ax[1, 1].set_xlabel("x coordinates (pixels)")
        ax[1, 1].set_ylabel("y coordinates (pixels)")
        ax[1, 1].legend(loc="upper right")
    else:
        ax[1, 1].axis("off")

    plt.tight_layout(rect=[0, 0.03, 1, 0.96])

    plt.show()

In [ ]:
# Center coordinates, filter low-confidence points, remove outliers, and interpolate
##########################################################################

# Detect and print confidence scores analysis (runs before any filtering)
#########

if not debug:
    print(
        "ℹ️ Debug output suppressed. Set debug=True to see detailed confidence score analysis."
    )

score_columns = [
    "left.score",
    "center.score",
    "right.score",
    "p1.score",
    "p2.score",
    "p3.score",
    "p4.score",
    "p5.score",
    "p6.score",
    "p7.score",
    "p8.score",
]

# VideoData1 confidence score analysis
if debug and "VideoData1_Has_Sleap" in globals() and VideoData1_Has_Sleap:
    lp.analyze_confidence_scores(
        VideoData1,
        score_columns,
        score_cutoff,
        get_eye_label("VideoData1"),
        debug=debug,
    )

# VideoData2 confidence score analysis
if debug and "VideoData2_Has_Sleap" in globals() and VideoData2_Has_Sleap:
    lp.analyze_confidence_scores(
        VideoData2,
        score_columns,
        score_cutoff,
        get_eye_label("VideoData2"),
        debug=debug,
    )


print()
print("=== Centering coordinates to the median pupil centre ===")
# Reset columns_of_interest to full coordinate column names (needed after
# QC plotting redefined it)
columns_of_interest = [
    "left.x",
    "left.y",
    "center.x",
    "center.y",
    "right.x",
    "right.y",
    "p1.x",
    "p1.y",
    "p2.x",
    "p2.y",
    "p3.x",
    "p3.y",
    "p4.x",
    "p4.y",
    "p5.x",
    "p5.y",
    "p6.x",
    "p6.y",
    "p7.x",
    "p7.y",
    "p8.x",
    "p8.y",
]
# VideoData1 processing
if "VideoData1_Has_Sleap" in globals() and VideoData1_Has_Sleap:
    VideoData1_centered = lp.center_coordinates_to_median(
        VideoData1, columns_of_interest, get_eye_label("VideoData1")
    )

# VideoData2 processing
if "VideoData2_Has_Sleap" in globals() and VideoData2_Has_Sleap:
    VideoData2_centered = lp.center_coordinates_to_median(
        VideoData2, columns_of_interest, get_eye_label("VideoData2")
    )

# remove low confidence points (score < threshold)
##################################################
if not NaNs_removed:
    if debug:
        print(
            "\n=== Score-based Filtering - point scores below threshold are replaced by interpolation ==="
        )
        print(f"Score threshold: {score_cutoff}")
    # List of point names (without .x, .y, .score)
    point_names = [
        "left",
        "right",
        "center",
        "p1",
        "p2",
        "p3",
        "p4",
        "p5",
        "p6",
        "p7",
        "p8",
    ]

    # VideoData1 score-based filtering
    if "VideoData1_Has_Sleap" in globals() and VideoData1_Has_Sleap:
        low_confidence_results_v1 = lp.filter_low_confidence_points(
            VideoData1,
            point_names,
            score_cutoff,
            get_eye_label("VideoData1"),
            debug=debug,
        )
        total_low_confidence_v1 = low_confidence_results_v1["total_low_score"]

    # VideoData2 score-based filtering
    if "VideoData2_Has_Sleap" in globals() and VideoData2_Has_Sleap:
        low_confidence_results_v2 = lp.filter_low_confidence_points(
            VideoData2,
            point_names,
            score_cutoff,
            get_eye_label("VideoData2"),
            debug=debug,
        )
        total_low_confidence_v2 = low_confidence_results_v2["total_low_score"]

    # Exclude poor-quality eye streams based on low-confidence interpolation burden.
    # This prevents downstream ellipse fitting failures on very sparse/invalid data.
    low_confidence_percentages = {"VideoData1": None, "VideoData2": None}
    excluded_videodata_due_to_low_confidence = []

    if "VideoData1_Has_Sleap" in globals() and VideoData1_Has_Sleap:
        total_possible_v1 = len(VideoData1) * len(point_names) * 2
        low_conf_pct_v1 = (
            (total_low_confidence_v1 / total_possible_v1) if total_possible_v1 > 0 else 1.0
        )
        low_confidence_percentages["VideoData1"] = low_conf_pct_v1
        if debug:
            print(
                f"{get_eye_label('VideoData1')} - Low-confidence interpolation burden: "
                f"{low_conf_pct_v1 * 100:.2f}% ({total_low_confidence_v1}/{total_possible_v1} coordinate values)"
            )
        if low_conf_pct_v1 > max_low_confidence_interpolation_fraction:
            VideoData1_Has_Sleap = False
            excluded_videodata_due_to_low_confidence.append("VideoData1")
            print(
                f"⚠️ {get_eye_label('VideoData1')} excluded from further processing: "
                f"low-confidence interpolation burden {low_conf_pct_v1 * 100:.2f}% exceeds "
                f"threshold {max_low_confidence_interpolation_fraction * 100:.2f}%"
            )

    if "VideoData2_Has_Sleap" in globals() and VideoData2_Has_Sleap:
        total_possible_v2 = len(VideoData2) * len(point_names) * 2
        low_conf_pct_v2 = (
            (total_low_confidence_v2 / total_possible_v2) if total_possible_v2 > 0 else 1.0
        )
        low_confidence_percentages["VideoData2"] = low_conf_pct_v2
        if debug:
            print(
                f"{get_eye_label('VideoData2')} - Low-confidence interpolation burden: "
                f"{low_conf_pct_v2 * 100:.2f}% ({total_low_confidence_v2}/{total_possible_v2} coordinate values)"
            )
        if low_conf_pct_v2 > max_low_confidence_interpolation_fraction:
            VideoData2_Has_Sleap = False
            excluded_videodata_due_to_low_confidence.append("VideoData2")
            print(
                f"⚠️ {get_eye_label('VideoData2')} excluded from further processing: "
                f"low-confidence interpolation burden {low_conf_pct_v2 * 100:.2f}% exceeds "
                f"threshold {max_low_confidence_interpolation_fraction * 100:.2f}%"
            )

    # remove outliers (x times SD)
    # then interpolates on all NaN values (skipped frames, low confidence inference points, outliers)
    ##########################################################################

    if debug:
        print(
            "\n=== Outlier Analysis - outlier points are replaced by interpolation ==="
        )

    # Reset columns_of_interest to full coordinate column names (needed after
    # QC plotting redefined it)
    columns_of_interest = [
        "left.x",
        "left.y",
        "center.x",
        "center.y",
        "right.x",
        "right.y",
        "p1.x",
        "p1.y",
        "p2.x",
        "p2.y",
        "p3.x",
        "p3.y",
        "p4.x",
        "p4.y",
        "p5.x",
        "p5.y",
        "p6.x",
        "p6.y",
        "p7.x",
        "p7.y",
        "p8.x",
        "p8.y",
    ]

    # VideoData1 outlier analysis and interpolation
    if "VideoData1_Has_Sleap" in globals() and VideoData1_Has_Sleap:
        outlier_results_v1 = lp.remove_outliers_and_interpolate(
            VideoData1,
            columns_of_interest,
            outlier_sd_threshold,
            get_eye_label("VideoData1"),
            debug=debug,
        )
        total_outliers_v1 = outlier_results_v1["total_outliers"]
        VideoData1 = outlier_results_v1["video_data_interpolated"]

    # VideoData2 outlier analysis and interpolation
    if "VideoData2_Has_Sleap" in globals() and VideoData2_Has_Sleap:
        outlier_results_v2 = lp.remove_outliers_and_interpolate(
            VideoData2,
            columns_of_interest,
            outlier_sd_threshold,
            get_eye_label("VideoData2"),
            debug=debug,
        )
        total_outliers_v2 = outlier_results_v2["total_outliers"]
        VideoData2 = outlier_results_v2["video_data_interpolated"]

    # Save per-video low-confidence totals and choose canonical eye with best
    # SLEAP inference quality. Tie rule: prefer VideoData2 due to generally
    # better effective resolution.
    low_confidence_counts = {
        "VideoData1": int(total_low_confidence_v1)
        if "total_low_confidence_v1" in globals()
        else None,
        "VideoData2": int(total_low_confidence_v2)
        if "total_low_confidence_v2" in globals()
        else None,
    }

    has_v1 = "VideoData1_Has_Sleap" in globals() and VideoData1_Has_Sleap
    has_v2 = "VideoData2_Has_Sleap" in globals() and VideoData2_Has_Sleap

    if has_v1 and not has_v2:
        eye_with_least_low_confidence = "VideoData1"
    elif has_v2 and not has_v1:
        eye_with_least_low_confidence = "VideoData2"
    elif has_v1 and has_v2:
        # Tie (>=) resolves to VideoData2 per acquisition-quality preference.
        eye_with_least_low_confidence = (
            "VideoData1"
            if low_confidence_counts["VideoData1"]
            < low_confidence_counts["VideoData2"]
            else "VideoData2"
        )
    else:
        eye_with_least_low_confidence = None

    # Set flag after both VideoData1 and VideoData2 processing is complete
    NaNs_removed = True
else:
    print("=== Interpolation already done, skipping ===")

In [ ]:
# Instance.score distribution and blink detection threshold (hard/adaptive hybrid)
##########################################################################
# When instance score is low, that's typically because of a blink or similar occlusion.
# Threshold can be either fixed (hard) or adaptive per video with hard clamps.

if not debug:
    print(
        "ℹ️ Debug output suppressed. Set debug=True to see detailed instance score distribution analysis."
    )

if debug:
    print("=" * 80)
    print("INSTANCE.SCORE DISTRIBUTION AND BLINK DETECTION THRESHOLD")
    print("=" * 80)
    mode_label = "adaptive-down-only" if (use_adaptive_blink_threshold and not adaptive_blink_allow_upward) else ("adaptive" if use_adaptive_blink_threshold else "hard")
    print(f"\nThreshold mode: {mode_label}")
    print(f"Base hard threshold: instance.score < {blink_instance_score_threshold}")
    if use_adaptive_blink_threshold:
        print(
            f"Adaptive settings: method={adaptive_blink_method}, percentile_fallback={adaptive_blink_percentile}, clamp=[{blink_threshold_min}, {blink_threshold_max}]"
        )
    print("=" * 80)

# Only analyze for dataset(s) that exist
has_v1 = "VideoData1_Has_Sleap" in globals() and VideoData1_Has_Sleap
has_v2 = "VideoData2_Has_Sleap" in globals() and VideoData2_Has_Sleap

# Get FPS for time calculations
fps_1_for_threshold = None
fps_2_for_threshold = None
if has_v1:
    fps_1_for_threshold = (
        FPS_1
        if "FPS_1" in globals()
        else (1 / VideoData1["Seconds"].diff().mean() if has_v1 else None)
    )
if has_v2:
    fps_2_for_threshold = (
        FPS_2
        if "FPS_2" in globals()
        else (1 / VideoData2["Seconds"].diff().mean() if has_v2 else None)
    )

# Compute per-video thresholds
blink_thresholds = {}
if has_v1:
    blink_thresholds["VideoData1"] = compute_blink_instance_score_threshold(
        VideoData1,
        hard_threshold=blink_instance_score_threshold,
        use_adaptive=use_adaptive_blink_threshold,
        adaptive_percentile=adaptive_blink_percentile,
        min_threshold=blink_threshold_min,
        max_threshold=blink_threshold_max,
        adaptive_method=adaptive_blink_method,
        allow_upward=adaptive_blink_allow_upward,
        min_peak_separation=adaptive_blink_min_peak_separation,
        min_low_fraction=adaptive_blink_min_low_fraction,
        max_low_fraction=adaptive_blink_max_low_fraction,
    )
if has_v2:
    blink_thresholds["VideoData2"] = compute_blink_instance_score_threshold(
        VideoData2,
        hard_threshold=blink_instance_score_threshold,
        use_adaptive=use_adaptive_blink_threshold,
        adaptive_percentile=adaptive_blink_percentile,
        min_threshold=blink_threshold_min,
        max_threshold=blink_threshold_max,
        adaptive_method=adaptive_blink_method,
        allow_upward=adaptive_blink_allow_upward,
        min_peak_separation=adaptive_blink_min_peak_separation,
        min_low_fraction=adaptive_blink_min_low_fraction,
        max_low_fraction=adaptive_blink_max_low_fraction,
    )

if debug:
    if "VideoData1" in blink_thresholds:
        print(
            f"{get_eye_label('VideoData1')} threshold: instance.score < {blink_thresholds['VideoData1']:.3f}"
        )
    if "VideoData2" in blink_thresholds:
        print(
            f"{get_eye_label('VideoData2')} threshold: instance.score < {blink_thresholds['VideoData2']:.3f}"
        )

# Plot combined histograms with per-video applied thresholds
if debug and (has_v1 or has_v2):
    lp.plot_instance_score_distributions_combined(
        VideoData1 if has_v1 else None,
        VideoData2 if has_v2 else None,
        blink_thresholds,
        has_v1=has_v1,
        has_v2=has_v2,
        threshold_label="Applied threshold",
    )

# Report statistics with per-video threshold
if has_v1:
    lp.analyze_instance_score_distribution(
        VideoData1,
        blink_thresholds["VideoData1"],
        fps_1_for_threshold,
        get_eye_label("VideoData1"),
        debug=debug,
        plot=False,
        threshold_label="Applied threshold",
    )

if has_v2:
    lp.analyze_instance_score_distribution(
        VideoData2,
        blink_thresholds["VideoData2"],
        fps_2_for_threshold,
        get_eye_label("VideoData2"),
        debug=debug,
        plot=False,
        threshold_label="Applied threshold",
    )

if debug:
    print(f"\n{'=' * 80}")
    print("Note: These thresholds will be used for blink detection in the next cell.")
    print("=" * 80)


In [ ]:
# Blink detection using instance.score - mark blinks and set coordinates to NaN (keep them as NaN, no interpolation)
##########################################################################

if not debug:
    print(
        "ℹ️ Debug output suppressed. Set debug=True to see detailed blink detection information."
    )

# Capture all print output to save to file


class TeeOutput:
    """Output to both stdout and a string buffer"""

    def __init__(self, stdout, buffer):
        self.stdout = stdout
        self.buffer = buffer

    def write(self, s):
        self.stdout.write(s)
        self.buffer.write(s)

    def flush(self):
        self.stdout.flush()
        self.buffer.flush()


output_buffer = io.StringIO()
original_stdout = sys.stdout
sys.stdout = TeeOutput(original_stdout, output_buffer)

# Run blink detection code with output captured
if "blink_thresholds" not in globals():
    blink_thresholds = {
        "VideoData1": blink_instance_score_threshold,
        "VideoData2": blink_instance_score_threshold,
    }

if debug:
    print("\n=== Blink Detection ===")

# VideoData1 blink detection
if "VideoData1_Has_Sleap" in globals() and VideoData1_Has_Sleap:
    # Get FPS if available, otherwise will be calculated in function
    fps_1 = FPS_1 if "FPS_1" in globals() else None

    # Get manual blinks if available
    manual_blinks_for_v1 = (
        manual_blinks_v1
        if "manual_blinks_v1" in globals() and manual_blinks_v1 is not None
        else None
    )

    # Run blink detection
    if debug:
        print(f"Using blink threshold for {get_eye_label('VideoData1')}: {blink_thresholds.get('VideoData1', blink_instance_score_threshold):.3f}")
    blink_results_v1 = lp.detect_blinks_for_video(
        video_data=VideoData1,
        columns_of_interest=columns_of_interest,
        blink_instance_score_threshold=blink_thresholds.get("VideoData1", blink_instance_score_threshold),
        long_blink_warning_ms=long_blink_warning_ms,
        min_frames_threshold=4,
        merge_window_frames=10,
        fps=fps_1,
        video_label=get_eye_label("VideoData1"),
        manual_blinks=manual_blinks_for_v1,
        debug=debug,
    )

    # Extract results to maintain compatibility with existing variable names
    blink_segments_v1 = blink_results_v1["blink_segments"]
    short_blink_segments_v1 = blink_results_v1["short_blink_segments"]
    blink_bouts_v1 = blink_results_v1["blink_bouts"]
    all_blink_segments_v1 = blink_results_v1["all_blink_segments"]
    fps_1 = blink_results_v1["fps"]  # Update fps_1 with calculated value
    FPS_1 = fps_1  # Also update global FPS_1 for use elsewhere
    long_blinks_warnings_v1 = blink_results_v1["long_blinks_warnings"]

# VideoData2 blink detection
if "VideoData2_Has_Sleap" in globals() and VideoData2_Has_Sleap:
    # Get FPS if available, otherwise will be calculated in function
    fps_2 = FPS_2 if "FPS_2" in globals() else None

    # Get manual blinks if available
    manual_blinks_for_v2 = (
        manual_blinks_v2
        if "manual_blinks_v2" in globals() and manual_blinks_v2 is not None
        else None
    )

    # Run blink detection
    if debug:
        print(f"Using blink threshold for {get_eye_label('VideoData2')}: {blink_thresholds.get('VideoData2', blink_instance_score_threshold):.3f}")
    blink_results_v2 = lp.detect_blinks_for_video(
        video_data=VideoData2,
        columns_of_interest=columns_of_interest,
        blink_instance_score_threshold=blink_thresholds.get("VideoData2", blink_instance_score_threshold),
        long_blink_warning_ms=long_blink_warning_ms,
        min_frames_threshold=4,
        merge_window_frames=10,
        fps=fps_2,
        video_label=get_eye_label("VideoData2"),
        manual_blinks=manual_blinks_for_v2,
        debug=debug,
    )

    # Extract results to maintain compatibility with existing variable names
    blink_segments_v2 = blink_results_v2["blink_segments"]
    short_blink_segments_v2 = blink_results_v2["short_blink_segments"]
    blink_bouts_v2 = blink_results_v2["blink_bouts"]
    all_blink_segments_v2 = blink_results_v2["all_blink_segments"]
    fps_2 = blink_results_v2["fps"]  # Update fps_2 with calculated value
    FPS_2 = fps_2  # Also update global FPS_2 for use elsewhere
    long_blinks_warnings_v2 = blink_results_v2["long_blinks_warnings"]

print("\n✅ Blink detection complete. Blink periods remain as NaN (not interpolated).")

# Compare blink bout timing between VideoData1 and VideoData2 (between eyes)
if (
    "VideoData1_Has_Sleap" in globals()
    and VideoData1_Has_Sleap
    and "VideoData2_Has_Sleap" in globals()
    and VideoData2_Has_Sleap
):
    if debug:
        print("\n" + "=" * 80)
        print("BLINK BOUT TIMING COMPARISON: VideoData1 vs VideoData2 (Between Eyes)")
        print("=" * 80)

    # Get blink bout frame ranges for both videos (if they exist)
    # Check if blink_bouts variables exist (they are created during blink
    # detection)
    try:
        has_bouts_v1 = "blink_bouts_v1" in globals() and len(blink_bouts_v1) > 0
    except BaseException:
        has_bouts_v1 = False

    try:
        has_bouts_v2 = "blink_bouts_v2" in globals() and len(blink_bouts_v2) > 0
    except BaseException:
        has_bouts_v2 = False

    if has_bouts_v1 and has_bouts_v2:
        # Convert bout indices to frame numbers
        bouts_v1 = []
        for i, bout in enumerate(blink_bouts_v1, 1):
            start_idx = bout["start_idx"]
            end_idx = bout["end_idx"]
            # CRITICAL FIX: Use stored frame_idx values if available (from merge_nearby_blinks)
            if "start_frame_idx" in bout and "end_frame_idx" in bout:
                start_frame = int(bout["start_frame_idx"]) if pd.notna(bout["start_frame_idx"]) else start_idx
                end_frame = int(bout["end_frame_idx"]) if pd.notna(bout["end_frame_idx"]) else end_idx
            elif "frame_idx" in VideoData1.columns:
                # Fallback: look up from DataFrame (may be wrong if DataFrame was modified)
                start_frame = int(VideoData1["frame_idx"].iloc[start_idx])
                end_frame = int(VideoData1["frame_idx"].iloc[end_idx])
            else:
                start_frame = start_idx
                end_frame = end_idx
            bouts_v1.append(
                {
                    "num": i,
                    "start_frame": start_frame,
                    "end_frame": end_frame,
                    "start_idx": start_idx,
                    "end_idx": end_idx,
                    "length": bout["length"],
                }
            )

        bouts_v2 = []
        for i, bout in enumerate(blink_bouts_v2, 1):
            start_idx = bout["start_idx"]
            end_idx = bout["end_idx"]
            # CRITICAL FIX: Use stored frame_idx values if available (from merge_nearby_blinks)
            if "start_frame_idx" in bout and "end_frame_idx" in bout:
                start_frame = int(bout["start_frame_idx"]) if pd.notna(bout["start_frame_idx"]) else start_idx
                end_frame = int(bout["end_frame_idx"]) if pd.notna(bout["end_frame_idx"]) else end_idx
            elif "frame_idx" in VideoData2.columns:
                # Fallback: look up from DataFrame (may be wrong if DataFrame was modified)
                start_frame = int(VideoData2["frame_idx"].iloc[start_idx])
                end_frame = int(VideoData2["frame_idx"].iloc[end_idx])
            else:
                start_frame = start_idx
                end_frame = end_idx
            bouts_v2.append(
                {
                    "num": i,
                    "start_frame": start_frame,
                    "end_frame": end_frame,
                    "start_idx": start_idx,
                    "end_idx": end_idx,
                    "length": bout["length"],
                }
            )

        # Find concurrent bouts (overlapping in time, synchronized by Seconds)
        concurrent_bouts = []
        v1_independent = []
        v2_independent = []

        v2_matched = set()  # Track which VideoData2 bouts have been matched

        for bout1 in bouts_v1:
            # Get time range for bout1
            v1_start_time = VideoData1["Seconds"].iloc[bout1["start_idx"]]
            v1_end_time = VideoData1["Seconds"].iloc[bout1["end_idx"]]

            found_match = False
            for bout2 in bouts_v2:
                # Get time range for bout2
                v2_start_time = VideoData2["Seconds"].iloc[bout2["start_idx"]]
                v2_end_time = VideoData2["Seconds"].iloc[bout2["end_idx"]]

                # Check if bouts overlap in time (any overlapping time period)
                overlap_start_time = max(v1_start_time, v2_start_time)
                overlap_end_time = min(v1_end_time, v2_end_time)

                if overlap_start_time <= overlap_end_time:
                    # Concurrent - they overlap in time
                    # Calculate overlap duration
                    overlap_duration = overlap_end_time - overlap_start_time

                    concurrent_bouts.append(
                        {
                            "v1_num": bout1["num"],
                            "v1_start_frame": bout1["start_frame"],
                            "v1_end_frame": bout1["end_frame"],
                            "v1_start_time": v1_start_time,
                            "v1_end_time": v1_end_time,
                            "v2_num": bout2["num"],
                            "v2_start_frame": bout2["start_frame"],
                            "v2_end_frame": bout2["end_frame"],
                            "v2_start_time": v2_start_time,
                            "v2_end_time": v2_end_time,
                            "overlap_start_time": overlap_start_time,
                            "overlap_end_time": overlap_end_time,
                            "overlap_duration": overlap_duration,
                        }
                    )
                    v2_matched.add(bout2["num"])
                    found_match = True
                    break

            if not found_match:
                v1_independent.append(bout1)

        # Find VideoData2 bouts that don't have matches
        for bout2 in bouts_v2:
            if bout2["num"] not in v2_matched:
                v2_independent.append(bout2)

        # Calculate statistics
        total_v1_bouts = len(bouts_v1)
        total_v2_bouts = len(bouts_v2)
        total_concurrent = len(concurrent_bouts)
        total_v1_independent = len(v1_independent)
        total_v2_independent = len(v2_independent)

        if debug:
            print("\nBlink bout counts:")
            print(f"  VideoData1: {total_v1_bouts} blink bout(s)")
            print(f"  VideoData2: {total_v2_bouts} blink bout(s)")
            print(f"  Concurrent: {total_concurrent} bout(s) (overlapping frames)")
            print(f"  VideoData1 only: {total_v1_independent} bout(s)")
            print(f"  VideoData2 only: {total_v2_independent} bout(s)")

            if total_v1_bouts > 0 and total_v2_bouts > 0:
                concurrent_pct_v1 = (total_concurrent / total_v1_bouts) * 100
                concurrent_pct_v2 = (total_concurrent / total_v2_bouts) * 100
                print("\nConcurrency percentage:")
                print(
                    f"  {concurrent_pct_v1:.1f}% of VideoData1 bouts are concurrent with VideoData2"
                )
                print(
                    f"  {concurrent_pct_v2:.1f}% of VideoData2 bouts are concurrent with VideoData1"
                )

                # Calculate timing offsets for concurrent bouts
                if len(concurrent_bouts) > 0:
                    time_offsets_ms = []
                    for cb in concurrent_bouts:
                        # Calculate offset from start times (already in
                        # Seconds)
                        offset_ms = (cb["v1_start_time"] - cb["v2_start_time"]) * 1000
                        time_offsets_ms.append(offset_ms)
                        cb["time_offset_ms"] = offset_ms

                    mean_offset = np.mean(time_offsets_ms)
                    std_offset = np.std(time_offsets_ms)
                    print("\nTiming offset for concurrent bouts:")
                    print(
                        f"  Mean offset (VideoData1 - VideoData2): {mean_offset:.2f} ms"
                    )
                    print(f"  Std offset: {std_offset:.2f} ms")
                    print(
                        f"  Range: {min(time_offsets_ms):.2f} to {max(time_offsets_ms):.2f} ms"
                    )

            # Visualization removed per request
            print("=" * 80)
    elif has_bouts_v1 or has_bouts_v2:
        print(
            "\n⚠️ Cannot compare blink bouts - only one eye has blink bouts detected:"
        )
        if has_bouts_v1:
            print(f"  VideoData1: {len(blink_bouts_v1)} blink bout(s)")
        else:
            print("  VideoData1: 0 blink bout(s)")
        if has_bouts_v2:
            print(f"  VideoData2: {len(blink_bouts_v2)} blink bout(s)")
        else:
            print("  VideoData2: 0 blink bout(s)")
    else:
        print("\n⚠️ Cannot compare blink bouts - neither video has blink bouts detected")

# Save blink detection results to CSV files
if "VideoData1_Has_Sleap" in globals() and VideoData1_Has_Sleap:
    if len(blink_segments_v1) > 0:
        # Collect blink information
        blink_data_v1 = []
        manual_blinks_for_csv = None
        if "manual_blinks_v1" in globals() and manual_blinks_v1 is not None:
            manual_blinks_for_csv = manual_blinks_v1

        for i, blink in enumerate(blink_segments_v1, 1):
            # CRITICAL FIX: Use stored frame_idx values from blink detection
            # These were captured before any DataFrame modifications (e.g., merge operations)
            if "start_frame_idx" in blink and "end_frame_idx" in blink:
                first_frame = int(blink["start_frame_idx"]) if pd.notna(blink["start_frame_idx"]) else blink["start_idx"]
                last_frame = int(blink["end_frame_idx"]) if pd.notna(blink["end_frame_idx"]) else blink["end_idx"]
            else:
                # Fallback to old method if frame_idx not stored (for backward compatibility)
                start_idx = blink["start_idx"]
                end_idx = blink["end_idx"]
                if "frame_idx" in VideoData1.columns:
                    first_frame = int(VideoData1["frame_idx"].iloc[start_idx])
                    last_frame = int(VideoData1["frame_idx"].iloc[end_idx])
                else:
                    first_frame = start_idx
                    last_frame = end_idx

            # Check if this blink matches a manual one (using function from
            # processing_functions)
            matches_manual = lp.check_manual_match(
                first_frame, last_frame, manual_blinks_for_csv
            )

            blink_data_v1.append(
                {
                    "blink_number": i,
                    "first_frame": first_frame,
                    "last_frame": last_frame,
                    "matches_manual": matches_manual,
                }
            )

        # Create DataFrame and save to CSV
        blink_df_v1 = pd.DataFrame(blink_data_v1)
        blink_csv_path_v1 = qc_debug_dir / "blink_detection_VideoData1.csv"
        blink_df_v1.to_csv(blink_csv_path_v1, index=False)
        print(
            f"\n✅ Blink detection results (VideoData1) saved to: {blink_csv_path_v1}"
        )
        print(f"   Saved {len(blink_data_v1)} blinks")

if "VideoData2_Has_Sleap" in globals() and VideoData2_Has_Sleap:
    if len(blink_segments_v2) > 0:
        # Collect blink information
        blink_data_v2 = []
        manual_blinks_for_csv = None
        if "manual_blinks_v2" in globals() and manual_blinks_v2 is not None:
            manual_blinks_for_csv = manual_blinks_v2

        for i, blink in enumerate(blink_segments_v2, 1):
            # CRITICAL FIX: Use stored frame_idx values from blink detection
            # These were captured before any DataFrame modifications (e.g., merge operations)
            if "start_frame_idx" in blink and "end_frame_idx" in blink:
                first_frame = int(blink["start_frame_idx"]) if pd.notna(blink["start_frame_idx"]) else blink["start_idx"]
                last_frame = int(blink["end_frame_idx"]) if pd.notna(blink["end_frame_idx"]) else blink["end_idx"]
            else:
                # Fallback to old method if frame_idx not stored (for backward compatibility)
                start_idx = blink["start_idx"]
                end_idx = blink["end_idx"]
                if "frame_idx" in VideoData2.columns:
                    first_frame = int(VideoData2["frame_idx"].iloc[start_idx])
                    last_frame = int(VideoData2["frame_idx"].iloc[end_idx])
                else:
                    first_frame = start_idx
                    last_frame = end_idx

            # Check if this blink matches a manual one (using function from
            # processing_functions)
            matches_manual = lp.check_manual_match(
                first_frame, last_frame, manual_blinks_for_csv
            )

            blink_data_v2.append(
                {
                    "blink_number": i,
                    "first_frame": first_frame,
                    "last_frame": last_frame,
                    "matches_manual": matches_manual,
                }
            )

        # Create DataFrame and save to CSV
        blink_df_v2 = pd.DataFrame(blink_data_v2)
        blink_csv_path_v2 = qc_debug_dir / "blink_detection_VideoData2.csv"
        blink_df_v2.to_csv(blink_csv_path_v2, index=False)
        print(
            f"\n✅ Blink detection results (VideoData2) saved to: {blink_csv_path_v2}"
        )
        print(f"   Saved {len(blink_data_v2)} blinks")

print("\n" + "=" * 80)
print("📹 MANUAL QC CHECK:")
print("=" * 80)
print("For instructions on how to prepare videos for manual blink detection QC,")
print("see: https://github.com/ranczlab/vestibular_vr_pipeline/issues/86")
print("=" * 80)

# Restore original stdout and save captured output to file
sys.stdout = original_stdout

# Get the captured output
captured_output = output_buffer.getvalue()

# Save to file in data_path folder
output_file = qc_debug_dir / "blink_detection_QC.txt"
output_file.parent.mkdir(parents=True, exist_ok=True)
with open(output_file, "w") as f:
    f.write(captured_output)

print(f"\n✅ Blink detection output saved to: {output_file}")

In [ ]:
# if needed, QC plot timeseries of interpolation corrected NaN in browser
#########################################################################
if plot_QC_timeseries:
    print("ℹ️ Figure opens in browser window, takes a bit of time.")

    # VideoData1 QC Plot
    if "VideoData1_Has_Sleap" in globals() and VideoData1_Has_Sleap:
        fig1 = make_subplots(
            rows=4,
            cols=1,
            shared_xaxes=True,
            vertical_spacing=0.05,
            subplot_titles=(
                "VideoData1 - X coordinates for pupil centre and left-right eye corner",
                "VideoData1 - Y coordinates for pupil centre and left-right eye corner",
                "VideoData1 - X coordinates for iris points",
                "VideoData1 - Y coordinates for iris points",
            ),
        )

        # Row 1: Plot left.x, center.x, right.x
        fig1.add_trace(
            go.Scatter(
                x=VideoData1["Seconds"],
                y=VideoData1["left.x"],
                mode="lines",
                name="left.x",
            ),
            row=1,
            col=1,
        )
        fig1.add_trace(
            go.Scatter(
                x=VideoData1["Seconds"],
                y=VideoData1["center.x"],
                mode="lines",
                name="center.x",
            ),
            row=1,
            col=1,
        )
        fig1.add_trace(
            go.Scatter(
                x=VideoData1["Seconds"],
                y=VideoData1["right.x"],
                mode="lines",
                name="right.x",
            ),
            row=1,
            col=1,
        )

        # Row 2: Plot left.y, center.y, right.y
        fig1.add_trace(
            go.Scatter(
                x=VideoData1["Seconds"],
                y=VideoData1["left.y"],
                mode="lines",
                name="left.y",
            ),
            row=2,
            col=1,
        )
        fig1.add_trace(
            go.Scatter(
                x=VideoData1["Seconds"],
                y=VideoData1["center.y"],
                mode="lines",
                name="center.y",
            ),
            row=2,
            col=1,
        )
        fig1.add_trace(
            go.Scatter(
                x=VideoData1["Seconds"],
                y=VideoData1["right.y"],
                mode="lines",
                name="right.y",
            ),
            row=2,
            col=1,
        )

        # Row 3: Plot p.x coordinates for p1 to p8
        for col in ["p1.x", "p2.x", "p3.x", "p4.x", "p5.x", "p6.x", "p7.x", "p8.x"]:
            fig1.add_trace(
                go.Scatter(
                    x=VideoData1["Seconds"], y=VideoData1[col], mode="lines", name=col
                ),
                row=3,
                col=1,
            )

        # Row 4: Plot p.y coordinates for p1 to p8
        for col in ["p1.y", "p2.y", "p3.y", "p4.y", "p5.y", "p6.y", "p7.y", "p8.y"]:
            fig1.add_trace(
                go.Scatter(
                    x=VideoData1["Seconds"], y=VideoData1[col], mode="lines", name=col
                ),
                row=4,
                col=1,
            )

        fig1.update_layout(
            height=1200,
            title_text="VideoData1 - Time series subplots for coordinates (QC after interpolation)",
            showlegend=True,
        )
        fig1.update_xaxes(title_text="Seconds", row=4, col=1)
        fig1.update_yaxes(title_text="X Position", row=1, col=1)
        fig1.update_yaxes(title_text="Y Position", row=2, col=1)
        fig1.update_yaxes(title_text="X Position", row=3, col=1)
        fig1.update_yaxes(title_text="Y Position", row=4, col=1)

        fig1.show(renderer="browser")

    # VideoData2 QC Plot
    if "VideoData2_Has_Sleap" in globals() and VideoData2_Has_Sleap:
        fig2 = make_subplots(
            rows=4,
            cols=1,
            shared_xaxes=True,
            vertical_spacing=0.05,
            subplot_titles=(
                "VideoData2 - X coordinates for pupil centre and left-right eye corner",
                "VideoData2 - Y coordinates for pupil centre and left-right eye corner",
                "VideoData2 - X coordinates for iris points",
                "VideoData2 - Y coordinates for iris points",
            ),
        )

        # Row 1: Plot left.x, center.x, right.x
        fig2.add_trace(
            go.Scatter(
                x=VideoData2["Seconds"],
                y=VideoData2["left.x"],
                mode="lines",
                name="left.x",
            ),
            row=1,
            col=1,
        )
        fig2.add_trace(
            go.Scatter(
                x=VideoData2["Seconds"],
                y=VideoData2["center.x"],
                mode="lines",
                name="center.x",
            ),
            row=1,
            col=1,
        )
        fig2.add_trace(
            go.Scatter(
                x=VideoData2["Seconds"],
                y=VideoData2["right.x"],
                mode="lines",
                name="right.x",
            ),
            row=1,
            col=1,
        )

        # Row 2: Plot left.y, center.y, right.y
        fig2.add_trace(
            go.Scatter(
                x=VideoData2["Seconds"],
                y=VideoData2["left.y"],
                mode="lines",
                name="left.y",
            ),
            row=2,
            col=1,
        )
        fig2.add_trace(
            go.Scatter(
                x=VideoData2["Seconds"],
                y=VideoData2["center.y"],
                mode="lines",
                name="center.y",
            ),
            row=2,
            col=1,
        )
        fig2.add_trace(
            go.Scatter(
                x=VideoData2["Seconds"],
                y=VideoData2["right.y"],
                mode="lines",
                name="right.y",
            ),
            row=2,
            col=1,
        )

        # Row 3: Plot p.x coordinates for p1 to p8

        for col in ["p1.x", "p2.x", "p3.x", "p4.x", "p5.x", "p6.x", "p7.x", "p8.x"]:
            fig2.add_trace(
                go.Scatter(
                    x=VideoData2["Seconds"], y=VideoData2[col], mode="lines", name=col
                ),
                row=3,
                col=1,
            )

        # Row 4: Plot p.y coordinates for p1 to p8

        for col in ["p1.y", "p2.y", "p3.y", "p4.y", "p5.y", "p6.y", "p7.y", "p8.y"]:
            fig2.add_trace(
                go.Scatter(
                    x=VideoData2["Seconds"], y=VideoData2[col], mode="lines", name=col
                ),
                row=4,
                col=1,
            )

        fig2.update_layout(
            height=1200,
            title_text="VideoData2 - Time series subplots for coordinates (QC after interpolation)",
            showlegend=True,
        )
        fig2.update_xaxes(title_text="Seconds", row=4, col=1)
        fig2.update_yaxes(title_text="X Position", row=1, col=1)
        fig2.update_yaxes(title_text="Y Position", row=2, col=1)
        fig2.update_yaxes(title_text="X Position", row=3, col=1)
        fig2.update_yaxes(title_text="Y Position", row=4, col=1)

        fig2.show(renderer="browser")

In [ ]:
# QC plot XY coordinate distributions after NaN are interpolated
################################################################

columns_of_interest = [
    "left.x",
    "left.y",
    "center.x",
    "center.y",
    "right.x",
    "right.y",
    "p1.x",
    "p1.y",
    "p2.x",
    "p2.y",
    "p3.x",
    "p3.y",
    "p4.x",
    "p4.y",
    "p5.x",
    "p5.y",
    "p6.x",
    "p6.y",
    "p7.x",
    "p7.y",
    "p8.x",
    "p8.y",
]

# Create coordinates_dict for both datasets
if VideoData1_Has_Sleap:
    coordinates_dict1_processed = lp.get_coordinates_dict(
        VideoData1, columns_of_interest
    )

if VideoData2_Has_Sleap:
    coordinates_dict2_processed = lp.get_coordinates_dict(
        VideoData2, columns_of_interest
    )


columns_of_interest = [
    "left",
    "right",
    "center",
    "p1",
    "p2",
    "p3",
    "p4",
    "p5",
    "p6",
    "p7",
    "p8",
]

# Filter out NaN values and calculate the min and max values for X and Y
# coordinates for both dict1 and dict2

def min_max_dict(coordinates_dict):
    x_min = min(
        [
            coordinates_dict[f"{col}.x"][~np.isnan(coordinates_dict[f"{col}.x"])].min()
            for col in columns_of_interest
        ]
    )

    x_max = max(
        [
            coordinates_dict[f"{col}.x"][~np.isnan(coordinates_dict[f"{col}.x"])].max()
            for col in columns_of_interest
        ]
    )

    y_min = min(
        [
            coordinates_dict[f"{col}.y"][~np.isnan(coordinates_dict[f"{col}.y"])].min()
            for col in columns_of_interest
        ]
    )

    y_max = max(
        [
            coordinates_dict[f"{col}.y"][~np.isnan(coordinates_dict[f"{col}.y"])].max()
            for col in columns_of_interest
        ]
    )

    return x_min, x_max, y_min, y_max


if plot_QC:
    if VideoData1_Has_Sleap:
        x_min1, x_max1, y_min1, y_max1 = min_max_dict(coordinates_dict1_processed)
    if VideoData2_Has_Sleap:
        x_min2, x_max2, y_min2, y_max2 = min_max_dict(coordinates_dict2_processed)

    # Use global min and max for consistency across subplots
    if VideoData1_Has_Sleap and VideoData2_Has_Sleap:
        x_min = min(x_min1, x_min2)
        x_max = max(x_max1, x_max2)
        y_min = min(y_min1, y_min2)
        y_max = max(y_max1, y_max2)
    elif VideoData1_Has_Sleap:
        x_min, x_max, y_min, y_max = x_min1, x_max1, y_min1, y_max1
    elif VideoData2_Has_Sleap:
        x_min, x_max, y_min, y_max = x_min2, x_max2, y_min2, y_max2

    fig, ax = plt.subplots(nrows=2, ncols=2, figsize=(18, 12))
    fig.suptitle(
        f"XY coordinate distribution of different points for {get_eye_label('VideoData1')} and {get_eye_label('VideoData2')} post outlier removal and NaN interpolation",
        fontsize=14,
    )

    # Define colormap for p1-p8
    colors = ["blue", "green", "red", "cyan", "magenta", "yellow", "black", "orange"]

    # Panel 1: left, right, center (VideoData1)
    if VideoData1_Has_Sleap:
        ax[0, 0].set_title(f"{get_eye_label('VideoData1')}: left, right, center")

        ax[0, 0].scatter(
            coordinates_dict1_processed["left.x"],
            coordinates_dict1_processed["left.y"],
            color="black",
            label="left",
            s=10,
        )

        ax[0, 0].scatter(
            coordinates_dict1_processed["right.x"],
            coordinates_dict1_processed["right.y"],
            color="grey",
            label="right",
            s=10,
        )

        ax[0, 0].scatter(
            coordinates_dict1_processed["center.x"],
            coordinates_dict1_processed["center.y"],
            color="red",
            label="center",
            s=10,
        )

        ax[0, 0].set_xlim([x_min, x_max])

        ax[0, 0].set_ylim([y_min, y_max])

        ax[0, 0].set_xlabel("x coordinates (pixels)")

        ax[0, 0].set_ylabel("y coordinates (pixels)")

        ax[0, 0].legend(loc="upper right")

        # Panel 2: p1 to p8 (VideoData1)

        ax[0, 1].set_title(f"{get_eye_label('VideoData1')}: p1 to p8")

        for idx, col in enumerate(columns_of_interest[3:]):
            ax[0, 1].scatter(
                coordinates_dict1_processed[f"{col}.x"],
                coordinates_dict1_processed[f"{col}.y"],
                color=colors[idx],
                label=col,
                s=5,
            )

        ax[0, 1].set_xlim([x_min, x_max])

        ax[0, 1].set_ylim([y_min, y_max])

        ax[0, 1].set_xlabel("x coordinates (pixels)")

        ax[0, 1].set_ylabel("y coordinates (pixels)")

        ax[0, 1].legend(loc="upper right")

    # Panel 3: left, right, center (VideoData2)
    if VideoData2_Has_Sleap:
        ax[1, 0].set_title(f"{get_eye_label('VideoData2')}: left, right, center")

        ax[1, 0].scatter(
            coordinates_dict2_processed["left.x"],
            coordinates_dict2_processed["left.y"],
            color="black",
            label="left",
            s=10,
        )

        ax[1, 0].scatter(
            coordinates_dict2_processed["right.x"],
            coordinates_dict2_processed["right.y"],
            color="grey",
            label="right",
            s=10,
        )

        ax[1, 0].scatter(
            coordinates_dict2_processed["center.x"],
            coordinates_dict2_processed["center.y"],
            color="red",
            label="center",
            s=10,
        )

        ax[1, 0].set_xlim([x_min, x_max])

        ax[1, 0].set_ylim([y_min, y_max])

        ax[1, 0].set_xlabel("x coordinates (pixels)")

        ax[1, 0].set_ylabel("y coordinates (pixels)")

        ax[1, 0].legend(loc="upper right")

        # Panel 4: p1 to p8 (VideoData2)

        ax[1, 1].set_title(f"{get_eye_label('VideoData2')}: p1 to p8")

        for idx, col in enumerate(columns_of_interest[3:]):
            ax[1, 1].scatter(
                coordinates_dict2_processed[f"{col}.x"],
                coordinates_dict2_processed[f"{col}.y"],
                color=colors[idx],
                label=col,
                s=5,
            )

        ax[1, 1].set_xlim([x_min, x_max])

        ax[1, 1].set_ylim([y_min, y_max])

        ax[1, 1].set_xlabel("x coordinates (pixels)")

        ax[1, 1].set_ylabel("y coordinates (pixels)")

        ax[1, 1].legend(loc="upper right")

    plt.tight_layout(rect=[0, 0.03, 1, 0.96])
    plt.show()

In [ ]:
# fit ellipses on the 8 points to determine pupil centre and diameter
##########################################################################

columns_of_interest = [
    "left.x",
    "left.y",
    "center.x",
    "center.y",
    "right.x",
    "right.y",
    "p1.x",
    "p1.y",
    "p2.x",
    "p2.y",
    "p3.x",
    "p3.y",
    "p4.x",
    "p4.y",
    "p5.x",
    "p5.y",
    "p6.x",
    "p6.y",
    "p7.x",
    "p7.y",
    "p8.x",
    "p8.y",
]

# VideoData1 processing
if VideoData1_Has_Sleap:
    print("=== VideoData1 Ellipse Fitting for Pupil Diameter ===")
    coordinates_dict1_processed = lp.get_coordinates_dict(
        VideoData1, columns_of_interest
    )

    theta1 = lp.find_horizontal_axis_angle(VideoData1, "left", "center")
    center_point1 = lp.get_left_right_center_point(coordinates_dict1_processed)

    columns_of_interest_reformatted = [
        "left",
        "right",
        "center",
        "p1",
        "p2",
        "p3",
        "p4",
        "p5",
        "p6",
        "p7",
        "p8",
    ]
    remformatted_coordinates_dict1 = lp.get_reformatted_coordinates_dict(
        coordinates_dict1_processed, columns_of_interest_reformatted
    )
    centered_coordinates_dict1 = lp.get_centered_coordinates_dict(
        remformatted_coordinates_dict1, center_point1
    )
    rotated_coordinates_dict1 = lp.get_rotated_coordinates_dict(
        centered_coordinates_dict1, theta1
    )

    columns_of_interest_ellipse = ["p1", "p2", "p3", "p4", "p5", "p6", "p7", "p8"]
    ellipse_parameters_data1, ellipse_center_points_data1 = (
        lp.get_fitted_ellipse_parameters(
            rotated_coordinates_dict1, columns_of_interest_ellipse
        )
    )

    average_diameter1 = np.mean(
        [ellipse_parameters_data1[:, 0], ellipse_parameters_data1[:, 1]], axis=0
    )

    SleapVideoData1 = process.convert_arrays_to_dataframe(
        [
            "Seconds",
            "Ellipse.Diameter",
            "Ellipse.Angle",
            "Ellipse.Center.X",
            "Ellipse.Center.Y",
        ],
        [
            VideoData1["Seconds"].values,
            average_diameter1,
            ellipse_parameters_data1[:, 2],
            ellipse_center_points_data1[:, 0],
            ellipse_center_points_data1[:, 1],
        ],
    )

# VideoData2 processing
if VideoData2_Has_Sleap:
    print("=== VideoData2 Ellipse Fitting for Pupil Diameter ===")
    coordinates_dict2_processed = lp.get_coordinates_dict(
        VideoData2, columns_of_interest
    )

    theta2 = lp.find_horizontal_axis_angle(VideoData2, "left", "center")
    center_point2 = lp.get_left_right_center_point(coordinates_dict2_processed)

    columns_of_interest_reformatted = [
        "left",
        "right",
        "center",
        "p1",
        "p2",
        "p3",
        "p4",
        "p5",
        "p6",
        "p7",
        "p8",
    ]
    remformatted_coordinates_dict2 = lp.get_reformatted_coordinates_dict(
        coordinates_dict2_processed, columns_of_interest_reformatted
    )
    centered_coordinates_dict2 = lp.get_centered_coordinates_dict(
        remformatted_coordinates_dict2, center_point2
    )
    rotated_coordinates_dict2 = lp.get_rotated_coordinates_dict(
        centered_coordinates_dict2, theta2
    )

    columns_of_interest_ellipse = ["p1", "p2", "p3", "p4", "p5", "p6", "p7", "p8"]
    ellipse_parameters_data2, ellipse_center_points_data2 = (
        lp.get_fitted_ellipse_parameters(
            rotated_coordinates_dict2, columns_of_interest_ellipse
        )
    )

    average_diameter2 = np.mean(
        [ellipse_parameters_data2[:, 0], ellipse_parameters_data2[:, 1]], axis=0
    )

    SleapVideoData2 = process.convert_arrays_to_dataframe(
        [
            "Seconds",
            "Ellipse.Diameter",
            "Ellipse.Angle",
            "Ellipse.Center.X",
            "Ellipse.Center.Y",
        ],
        [
            VideoData2["Seconds"].values,
            average_diameter2,
            ellipse_parameters_data2[:, 2],
            ellipse_center_points_data2[:, 0],
            ellipse_center_points_data2[:, 1],
        ],
    )


# Filter pupil diameter using a user set (default 10) Hz Butterworth low-pass filter
##########################################################################

# VideoData1 filtering
if VideoData1_Has_Sleap:
    print("\n=== Filtering pupil diameter for VideoData1  ===")
    # Butterworth filter parameters - pupil_filter_cutoff_hz low-pass filter
    # Sampling frequency (Hz)
    fs1 = 1 / np.median(np.diff(SleapVideoData1["Seconds"]))
    order = pupil_filter_order

    b1, a1 = butter(order, pupil_filter_cutoff_hz / (0.5 * fs1), btype="low")

    # Handle NaN values before filtering (from blink detection)
    # Replace NaN with forward-fill for filtering purposes only (to avoid
    # filtfilt issues)
    diameter_data = SleapVideoData1["Ellipse.Diameter"].copy()
    # Use ffill() and bfill() instead of deprecated fillna(method='ffill')
    diameter_data_filled = diameter_data.ffill().bfill()

    # Apply Butterworth filter
    if not diameter_data_filled.isna().all():
        filtered = filtfilt(b1, a1, diameter_data_filled)
        # Restore NaN values at original NaN positions (from blinks)
        filtered = pd.Series(filtered, index=diameter_data.index)
        filtered[diameter_data.isna()] = np.nan
        SleapVideoData1["Ellipse.Diameter.Filt"] = filtered
    else:
        # If all values are NaN, just copy
        SleapVideoData1["Ellipse.Diameter.Filt"] = diameter_data

# VideoData2 filtering
if VideoData2_Has_Sleap:
    print("=== Filtering pupil diameter for VideoData2 ===")
    # Butterworth filter parameters - pupil_filter_cutoff_hz low-pass filter
    # Sampling frequency (Hz)
    fs2 = 1 / np.median(np.diff(SleapVideoData2["Seconds"]))
    order = pupil_filter_order

    b2, a2 = butter(order, pupil_filter_cutoff_hz / (0.5 * fs2), btype="low")

    # Handle NaN values before filtering (from blink detection)
    # Replace NaN with forward-fill for filtering purposes only (to avoid
    # filtfilt issues)
    diameter_data = SleapVideoData2["Ellipse.Diameter"].copy()
    # Use ffill() and bfill() instead of deprecated fillna(method='ffill')
    diameter_data_filled = diameter_data.ffill().bfill()

    # Apply Butterworth filter
    if not diameter_data_filled.isna().all():
        filtered = filtfilt(b2, a2, diameter_data_filled)
        # Restore NaN values at original NaN positions (from blinks)
        filtered = pd.Series(filtered, index=diameter_data.index)
        filtered[diameter_data.isna()] = np.nan
        SleapVideoData2["Ellipse.Diameter.Filt"] = filtered
    else:
        # If all values are NaN, just copy
        SleapVideoData2["Ellipse.Diameter.Filt"] = diameter_data

print("✅ Done calculating pupil diameter and angle for both VideoData1 and VideoData2")

In [ ]:
# cross-correlate pupil diameter for left and right eye
##########################################################################

if VideoData1_Has_Sleap and VideoData2_Has_Sleap:
    # Cross-correlation analysis
    print("=== Cross-Correlation Analysis ===")

    # Get pupil diameter data
    # Use filtered diameter data (with NaN restored at blink positions)
    pupil1 = SleapVideoData1["Ellipse.Diameter.Filt"].values
    pupil2 = SleapVideoData2["Ellipse.Diameter.Filt"].values

    # Handle different lengths by using the shorter dataset length
    min_length = min(len(pupil1), len(pupil2))

    # Truncate both datasets to the same length (preserving time alignment)
    pupil1_truncated = pupil1[:min_length]
    pupil2_truncated = pupil2[:min_length]

    # Remove NaN values for correlation - preserve time alignment by only keeping pairs where BOTH are valid
    # This ensures cross-correlation is computed on temporally aligned data
    valid_mask1 = ~np.isnan(pupil1_truncated)
    valid_mask2 = ~np.isnan(pupil2_truncated)
    # Only use indices where both arrays have valid data
    valid_mask = valid_mask1 & valid_mask2

    # Extract aligned pairs (preserves temporal alignment)
    pupil1_clean = pupil1_truncated[valid_mask]
    pupil2_clean = pupil2_truncated[valid_mask]

    # Check if we have enough data
    if len(pupil1_clean) < 2 or len(pupil2_clean) < 2:
        print("❌ Error: Not enough valid data points for correlation analysis")
    else:
        # Z-score normalize both signals before cross-correlation
        # This accounts for different camera magnifications/orientations by comparing relative changes
        # Formula: z = (x - mean) / std
        pupil1_mean = np.mean(pupil1_clean)
        pupil1_std = np.std(pupil1_clean)
        pupil2_mean = np.mean(pupil2_clean)
        pupil2_std = np.std(pupil2_clean)

        if pupil1_std > 0 and pupil2_std > 0:
            pupil1_z = (pupil1_clean - pupil1_mean) / pupil1_std
            pupil2_z = (pupil2_clean - pupil2_mean) / pupil2_std
            print(
                "Applied z-score normalization to pupil diameter signals (accounts for different camera magnifications)"
            )
            print(f"  VideoData1: mean={pupil1_mean:.2f}, std={pupil1_std:.2f}")
            print(f"  VideoData2: mean={pupil2_mean:.2f}, std={pupil2_std:.2f}")
        else:
            print(
                "⚠️ Warning: Zero variance detected, using raw signals (no normalization)"
            )
            pupil1_z = pupil1_clean
            pupil2_z = pupil2_clean

        # Calculate cross-correlation using z-scored signals
        try:
            correlation = correlate(pupil1_z, pupil2_z, mode="full")

            # Calculate lags (in samples)
            lags = np.arange(-len(pupil2_z) + 1, len(pupil1_z))

            # Convert lags to time (assuming same sampling rate)
            dt = np.median(np.diff(SleapVideoData1["Seconds"]))
            lag_times = lags * dt

            # Find peak correlation and corresponding lag
            peak_idx = np.argmax(correlation)
            peak_correlation = correlation[peak_idx]
            peak_lag_samples = lags[peak_idx]
            peak_lag_time = lag_times[peak_idx]
            peak_lag_time_display = peak_lag_time  # for final QC figure

            print(f"Peak lag (time): {peak_lag_time:.4f} seconds")

            # Normalize correlation to [-1, 1] range (for z-scored signals,
            # this is standard normalization)
            norm_factor = np.sqrt(np.sum(pupil1_z**2) * np.sum(pupil2_z**2))
            if norm_factor > 0:
                correlation_normalized = correlation / norm_factor
                peak_correlation_normalized = correlation_normalized[peak_idx]
                print(f"Peak normalized correlation: {peak_correlation_normalized:.4f}")
            else:
                print("❌ Error: Cannot normalize correlation (zero variance)")
                correlation_normalized = correlation
                peak_correlation_normalized = 0

        except Exception as e:
            print(f"❌ Error in cross-correlation calculation: {e}")

    # Additional correlation statistics
    if len(pupil1_clean) >= 2 and len(pupil2_clean) >= 2:
        try:
            # Calculate Pearson correlation coefficient on z-scored signals
            # Note: For z-scored signals, Pearson correlation is equivalent to
            # the normalized cross-correlation at zero lag
            pearson_r, pearson_p = pearsonr(pupil1_z, pupil2_z)
            pearson_r_display = pearson_r
            pearson_p_display = pearson_p

            print("\n=== Additional Statistics ===")
            print(f"Pearson correlation coefficient: {pearson_r:.2f}")

            # Handle extremely small p-values
            if pearson_p < 1e-300:
                print("Pearson p-value: < 1e-300 (extremely significant)")
            else:
                print(f"Pearson p-value: {pearson_p:.5e}")

        except Exception as e:
            print(f"❌ Error in additional statistics: {e}")
            pearson_r_display = None
            pearson_p_display = None
    else:
        print("❌ Cannot calculate additional statistics - insufficient data")
else:
    print("Only one eye is present, no pupil diameter cross-correlation can be done")

In [ ]:
# check if Second values match 1:1 between VideoData and SleapVideoData then merge them into VideoData
##########################################################################

if VideoData1_Has_Sleap is True:
    if VideoData1["Seconds"].equals(SleapVideoData1["Seconds"]) is False:
        print(
            f"❗ {get_eye_label('VideoData1')}: The 'Seconds' columns DO NOT correspond 1:1 between the two DataFrames. This should not happen"
        )
    else:
        VideoData1 = VideoData1.merge(SleapVideoData1, on="Seconds", how="outer")
        del SleapVideoData1
        # CRITICAL FIX: Validate frame_idx after merge
        if "frame_idx" in VideoData1.columns:
            if VideoData1['frame_idx'].duplicated().any():
                dup_count = VideoData1['frame_idx'].duplicated().sum()
                print(f"   ⚠️ WARNING: VideoData1 has {dup_count} duplicate frame_idx values after merge with SleapVideoData1!")
                print(f"      This should not happen. Frame_idx should be unique.")
            if not VideoData1['frame_idx'].is_monotonic_increasing:
                print(f"   ⚠️ WARNING: VideoData1 frame_idx is not monotonic after merge with SleapVideoData1!")
                print(f"      This should not happen. Frame_idx should be strictly increasing.")

if VideoData2_Has_Sleap is True:
    if VideoData2["Seconds"].equals(SleapVideoData2["Seconds"]) is False:
        print(
            f"❗ {get_eye_label('VideoData2')}: The 'Seconds' columns DO NOT correspond 1:1 between the two DataFrames. This should not happen"
        )
    else:
        VideoData2 = VideoData2.merge(SleapVideoData2, on="Seconds", how="outer")
        del SleapVideoData2
        # CRITICAL FIX: Validate frame_idx after merge
        if "frame_idx" in VideoData2.columns:
            if VideoData2['frame_idx'].duplicated().any():
                dup_count = VideoData2['frame_idx'].duplicated().sum()
                print(f"   ⚠️ WARNING: VideoData2 has {dup_count} duplicate frame_idx values after merge with SleapVideoData2!")
                print(f"      This should not happen. Frame_idx should be unique.")
            if not VideoData2['frame_idx'].is_monotonic_increasing:
                print(f"   ⚠️ WARNING: VideoData2 frame_idx is not monotonic after merge with SleapVideoData2!")
                print(f"      This should not happen. Frame_idx should be strictly increasing.")
gc.collect()
None

In [ ]:
# Compare SLEAP center.x and .y with fitted ellipse centre distributions for both VideoData1 and VideoData2
##########################################################################

# ------------------------------------------------------------------
# 1) Compute correlations for VideoData1
# ------------------------------------------------------------------
if VideoData1_Has_Sleap is True:
    print("=== VideoData1 Analysis ===")
    slope_x1, intercept_x1, r_value_x1, p_value_x1, std_err_x1 = linregress(
        VideoData1["Ellipse.Center.X"], VideoData1["center.x"]
    )
    r_squared_x1 = r_value_x1**2
    print(
        f"{get_eye_label('VideoData1')} - R^2 between center point and ellipse center X data: {r_squared_x1:.4f}"
    )

    slope_y1, intercept_y1, r_value_y1, p_value_y1, std_err_y1 = linregress(
        VideoData1["Ellipse.Center.Y"], VideoData1["center.y"]
    )
    r_squared_y1 = r_value_y1**2
    print(
        f"{get_eye_label('VideoData1')} - R^2 between center point and ellipse center Y data: {r_squared_y1:.4f}"
    )

# ------------------------------------------------------------------
# 2) Compute correlations for VideoData2
# ------------------------------------------------------------------
if VideoData2_Has_Sleap is True:
    print("\n=== VideoData2 Analysis ===")
    slope_x2, intercept_x2, r_value_x2, p_value_x2, std_err_x2 = linregress(
        VideoData2["Ellipse.Center.X"], VideoData2["center.x"]
    )
    r_squared_x2 = r_value_x2**2
    print(
        f"{get_eye_label('VideoData2')} - R^2 between center point and ellipse center X data: {r_squared_x2:.4f}"
    )

    slope_y2, intercept_y2, r_value_y2, p_value_y2, std_err_y2 = linregress(
        VideoData2["Ellipse.Center.Y"], VideoData2["center.y"]
    )
    r_squared_y2 = r_value_y2**2
    print(
        f"{get_eye_label('VideoData2')} - R^2 between center point and ellipse center Y data: {r_squared_y2:.4f}"
    )

# ------------------------------------------------------------------
# 3) Center of Mass Analysis (if both VideoData1 and VideoData2 are available)
# ------------------------------------------------------------------
if VideoData1_Has_Sleap is True and VideoData2_Has_Sleap is True:
    print("\n=== Center of Mass Distance Analysis ===")

    # Calculate center of mass (mean) for VideoData1
    com_center_x1 = VideoData1["center.x"].mean()
    com_center_y1 = VideoData1["center.y"].mean()
    com_ellipse_x1 = VideoData1["Ellipse.Center.X"].mean()
    com_ellipse_y1 = VideoData1["Ellipse.Center.Y"].mean()

    # Calculate absolute distances for VideoData1

    dist_x1 = abs(com_center_x1 - com_ellipse_x1)

    dist_y1 = abs(com_center_y1 - com_ellipse_y1)

    print(f"\n{get_eye_label('VideoData1')}:")

    print(
        f"  Center of mass for center.x/y: ({com_center_x1:.4f}, {com_center_y1:.4f})"
    )

    print(
        f"  Center of mass for Ellipse.Center.X/Y: ({com_ellipse_x1:.4f}, {com_ellipse_y1:.4f})"
    )

    print(f"  Absolute distance in X: {dist_x1:.4f} pixels")

    print(f"  Absolute distance in Y: {dist_y1:.4f} pixels")

    # Calculate center of mass (mean) for VideoData2

    com_center_x2 = VideoData2["center.x"].mean()

    com_center_y2 = VideoData2["center.y"].mean()

    com_ellipse_x2 = VideoData2["Ellipse.Center.X"].mean()

    com_ellipse_y2 = VideoData2["Ellipse.Center.Y"].mean()

    # Calculate absolute distances for VideoData2

    dist_x2 = abs(com_center_x2 - com_ellipse_x2)

    dist_y2 = abs(com_center_y2 - com_ellipse_y2)

    print(f"\n{get_eye_label('VideoData2')}:")

    print(
        f"  Center of mass for center.x/y: ({com_center_x2:.4f}, {com_center_y2:.4f})"
    )

    print(
        f"  Center of mass for Ellipse.Center.X/Y: ({com_ellipse_x2:.4f}, {com_ellipse_y2:.4f})"
    )

    print(f"  Absolute distance in X: {dist_x2:.4f} pixels")

    print(f"  Absolute distance in Y: {dist_y2:.4f} pixels")


# ------------------------------------------------------------------
# 4) Re-center Ellipse.Center.X and Ellipse.Center.Y using median
# ------------------------------------------------------------------
print("\n=== Re-centering Ellipse.Center coordinates ===")


# Re-center VideoData1 Ellipse.Center coordinates

if VideoData1_Has_Sleap is True:
    # Calculate median
    median_ellipse_x1 = VideoData1["Ellipse.Center.X"].median()
    median_ellipse_y1 = VideoData1["Ellipse.Center.Y"].median()

    # Center the coordinates
    VideoData1["Ellipse.Center.X"] = VideoData1["Ellipse.Center.X"] - median_ellipse_x1
    VideoData1["Ellipse.Center.Y"] = VideoData1["Ellipse.Center.Y"] - median_ellipse_y1

    print(
        f"{get_eye_label('VideoData1')} - Re-centered Ellipse.Center using median: ({median_ellipse_x1:.4f}, {median_ellipse_y1:.4f})"
    )

# Re-center VideoData2 Ellipse.Center coordinates
if VideoData2_Has_Sleap is True:
    # Calculate median
    median_ellipse_x2 = VideoData2["Ellipse.Center.X"].median()
    median_ellipse_y2 = VideoData2["Ellipse.Center.Y"].median()

    # Center the coordinates
    VideoData2["Ellipse.Center.X"] = VideoData2["Ellipse.Center.X"] - median_ellipse_x2
    VideoData2["Ellipse.Center.Y"] = VideoData2["Ellipse.Center.Y"] - median_ellipse_y2

    print(
        f"{get_eye_label('VideoData2')} - Re-centered Ellipse.Center using median: ({median_ellipse_x2:.4f}, {median_ellipse_y2:.4f})"
    )

In [ ]:
# Make and save summary QC plot using matplotlib with scatter plots for 2D
# distributions

# Initialize the statistics variables (these are calculated in Cell 11)
try:
    pearson_r_display
except NameError:
    pearson_r_display = None
    pearson_p_display = None
    peak_lag_time_display = None
    print("⚠️ Note: Statistics not found. They should be calculated in Cell 11.")

# Visualization and statistics calculation (only if plot_QC is True)
if plot_QC:
    # Calculate correlation for Ellipse.Center.X between VideoData1 and
    # VideoData2 (if both exist)
    pearson_r_center = None
    pearson_p_center = None
    peak_lag_time_center = None
else:
    pearson_r_center = None
    pearson_p_center = None
    peak_lag_time_center = None


if VideoData1_Has_Sleap and VideoData2_Has_Sleap:
    # Get the Center.X data
    center_x1 = VideoData1["Ellipse.Center.X"].values
    center_x2 = VideoData2["Ellipse.Center.X"].values
    min_length = min(len(center_x1), len(center_x2))
    center_x1_truncated = center_x1[:min_length]
    center_x2_truncated = center_x2[:min_length]
    valid_mask1 = ~np.isnan(center_x1_truncated)
    valid_mask2 = ~np.isnan(center_x2_truncated)
    valid_mask = valid_mask1 & valid_mask2
    center_x1_clean = center_x1_truncated[valid_mask]
    center_x2_clean = center_x2_truncated[valid_mask]

    if len(center_x1_clean) >= 2 and len(center_x2_clean) >= 2:
        try:
            # Calculate Pearson correlation
            pearson_r_center, pearson_p_center = pearsonr(
                center_x1_clean, center_x2_clean
            )
            # Calculate cross-correlation for peak lag
            correlation = correlate(center_x1_clean, center_x2_clean, mode="full")
            lags = np.arange(-len(center_x2_clean) + 1, len(center_x1_clean))
            dt = np.median(np.diff(VideoData1["Seconds"]))
            lag_times = lags * dt
            peak_idx = np.argmax(correlation)
            peak_lag_time_center = lag_times[peak_idx]
        except Exception as e:
            print(f"❌ Error calculating Ellipse.Center.X correlation stats: {e}")


# Create the QC summary figure using matplotlib with custom grid layout
fig = plt.figure(figsize=(20, 18))
fig.suptitle(str(data_path), fontsize=16, y=0.995)

# Create a grid layout:
# - Top row (full width): VideoData1 Time Series
# - Second row (full width): VideoData2 Time Series
# - Third row (two columns): 2D scatter plots (VideoData1 left, VideoData2 right)
# - Fourth row (two columns): Pupil diameter (left), Ellipse.Center.X correlation (right)

gs = fig.add_gridspec(4, 2, hspace=0.3, wspace=0.3)

# Panel 1: VideoData1 center coordinates - Time Series (full width)
if VideoData1_Has_Sleap:
    ax1 = fig.add_subplot(gs[0, :])
    ax1.plot(
        VideoData1_centered["Seconds"],
        VideoData1_centered["center.x"],
        linewidth=0.5,
        c="blue",
        alpha=0.6,
        label="center.x original",
    )
    ax1.plot(
        VideoData1["Seconds"],
        VideoData1["Ellipse.Center.X"],
        linewidth=0.5,
        c="red",
        alpha=0.6,
        label="Ellipse Center.X",
    )
    ax1.set_xlabel("Time (s)")
    ax1.set_ylabel("Position (pixels)")
    ax1.set_title(f"{get_eye_label('VideoData1')} - center.X Time Series")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

# Panel 2: VideoData2 center coordinates - Time Series (full width)
if VideoData2_Has_Sleap:
    ax2 = fig.add_subplot(gs[1, :])
    ax2.plot(
        VideoData2_centered["Seconds"],
        VideoData2_centered["center.x"],
        linewidth=0.5,
        c="blue",
        alpha=0.6,
        label="center.x original",
    )
    ax2.plot(
        VideoData2["Seconds"],
        VideoData2["Ellipse.Center.X"],
        linewidth=0.5,
        c="red",
        alpha=0.6,
        label="Ellipse Center.X",
    )
    ax2.set_xlabel("Time (s)")
    ax2.set_ylabel("Position (pixels)")
    ax2.set_title(f"{get_eye_label('VideoData2')} - center.X Time Series")
    ax2.legend()
    ax2.grid(True, alpha=0.3)

# Panel 3: VideoData1 center coordinates - Scatter plot (left half)
if VideoData1_Has_Sleap:
    ax3 = fig.add_subplot(gs[2, 0])

    # Ellipse.Center (blue)
    x_ellipse1 = VideoData1["Ellipse.Center.X"].to_numpy()
    y_ellipse1 = VideoData1["Ellipse.Center.Y"].to_numpy()
    mask1 = ~(np.isnan(x_ellipse1) | np.isnan(y_ellipse1))

    ax3.scatter(
        x_ellipse1[mask1],
        y_ellipse1[mask1],
        s=1,
        alpha=0.3,
        c="blue",
        label="Ellipse.Center",
    )

    # Center (red) - from centered data
    x_center1 = VideoData1_centered["center.x"].to_numpy()
    y_center1 = VideoData1_centered["center.y"].to_numpy()
    mask2 = ~(np.isnan(x_center1) | np.isnan(y_center1))

    ax3.scatter(
        x_center1[mask2],
        y_center1[mask2],
        s=1,
        alpha=0.3,
        c="red",
        label="center.x original",
    )

    ax3.set_xlabel("Center X (pixels)")
    ax3.set_ylabel("Center Y (pixels)")
    ax3.set_title(
        f"{get_eye_label('VideoData1')} - Center X-Y Distribution (center.X vs Ellipse)"
    )
    ax3.legend()
    ax3.grid(True, alpha=0.3)

    # Add R² statistics for VideoData1 (bottom left)
    try:
        if "r_squared_x1" in globals() and "r_squared_y1" in globals():
            stats_text = f"R² X: {r_squared_x1:.2g}\nR² Y: {r_squared_y1:.2g}"
            ax3.text(
                0.02,
                0.02,
                stats_text,
                transform=ax3.transAxes,
                verticalalignment="bottom",
                horizontalalignment="left",
                bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.8),
                fontsize=9,
                family="monospace",
            )
    except BaseException:
        pass

    # Add center of mass distance for VideoData1 (bottom right)
    try:
        if "dist_x1" in globals() and "dist_y1" in globals():
            distance_text = f"COM Dist X: {dist_x1:.3g}\nCOM Dist Y: {dist_y1:.3g}"
            ax3.text(
                0.98,
                0.02,
                distance_text,
                transform=ax3.transAxes,
                verticalalignment="bottom",
                horizontalalignment="right",
                bbox=dict(boxstyle="round", facecolor="lightblue", alpha=0.8),
                fontsize=9,
                family="monospace",
            )
    except BaseException:
        pass

# Panel 4: VideoData2 center coordinates - Scatter plot (right half)
if VideoData2_Has_Sleap:
    ax4 = fig.add_subplot(gs[2, 1])

    # Ellipse.Center (blue)
    x_ellipse2 = VideoData2["Ellipse.Center.X"].to_numpy()
    y_ellipse2 = VideoData2["Ellipse.Center.Y"].to_numpy()
    mask3 = ~(np.isnan(x_ellipse2) | np.isnan(y_ellipse2))

    ax4.scatter(
        x_ellipse2[mask3],
        y_ellipse2[mask3],
        s=1,
        alpha=0.3,
        c="blue",
        label="Ellipse.Center",
    )

    # Center (red) - from centered data
    x_center2 = VideoData2_centered["center.x"].to_numpy()
    y_center2 = VideoData2_centered["center.y"].to_numpy()
    mask4 = ~(np.isnan(x_center2) | np.isnan(y_center2))

    ax4.scatter(
        x_center2[mask4],
        y_center2[mask4],
        s=1,
        alpha=0.3,
        c="red",
        label="center.X Center",
    )

    ax4.set_xlabel("Center X (pixels)")
    ax4.set_ylabel("Center Y (pixels)")
    ax4.set_title(
        f"{get_eye_label('VideoData2')} - Center X-Y Distribution (center.X vs Ellipse)"
    )
    ax4.legend()
    ax4.grid(True, alpha=0.3)

    # Add R² statistics for VideoData2 (bottom left)
    try:
        if "r_squared_x2" in globals() and "r_squared_y2" in globals():
            stats_text = f"R² X: {r_squared_x2:.2g}\nR² Y: {r_squared_y2:.2g}"
            ax4.text(
                0.02,
                0.02,
                stats_text,
                transform=ax4.transAxes,
                verticalalignment="bottom",
                horizontalalignment="left",
                bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.8),
                fontsize=9,
                family="monospace",
            )
    except BaseException:
        pass

    # Add center of mass distance for VideoData2 (bottom right)
    try:
        if "dist_x2" in globals() and "dist_y2" in globals():
            distance_text = f"COM Dist X: {dist_x2:.3g}\nCOM Dist Y: {dist_y2:.3g}"
            ax4.text(
                0.98,
                0.02,
                distance_text,
                transform=ax4.transAxes,
                verticalalignment="bottom",
                horizontalalignment="right",
                bbox=dict(boxstyle="round", facecolor="lightblue", alpha=0.8),
                fontsize=9,
                family="monospace",
            )
    except BaseException:
        pass

# Panel 5: Pupil diameter comparison (bottom left)
ax5 = fig.add_subplot(gs[3, 0])
if VideoData1_Has_Sleap and VideoData2_Has_Sleap:
    ax5.plot(
        VideoData1["Seconds"],
        VideoData1["Ellipse.Diameter.Filt"],
        linewidth=0.5,
        c="#FF7F00",
        alpha=0.6,
        label="VideoData1 Diameter",
    )
    ax5.plot(
        VideoData2["Seconds"],
        VideoData2["Ellipse.Diameter.Filt"],
        linewidth=0.5,
        c="#9370DB",
        alpha=0.6,
        label="VideoData2 Diameter",
    )
elif VideoData1_Has_Sleap:
    ax5.plot(
        VideoData1["Seconds"],
        VideoData1["Ellipse.Diameter.Filt"],
        linewidth=0.5,
        c="#FF7F00",
        alpha=0.6,
        label="VideoData1 Diameter",
    )
elif VideoData2_Has_Sleap:
    ax5.plot(
        VideoData2["Seconds"],
        VideoData2["Ellipse.Diameter.Filt"],
        linewidth=0.5,
        c="#9370DB",
        alpha=0.6,
        label="VideoData2 Diameter",
    )

ax5.set_xlabel("Time (s)")
ax5.set_ylabel("Diameter (pixels)")
ax5.set_title("Pupil Diameter Comparison")
ax5.legend()
ax5.grid(True, alpha=0.3)

# Add statistics text to Panel 5
if (
    pearson_r_display is not None
    and pearson_p_display is not None
    and peak_lag_time_display is not None
):
    stats_text = (
        f"Pearson r = {pearson_r_display:.4f}\n"
        f"Pearson p = {pearson_p_display:.4e}\n"
        f"Peak lag = {peak_lag_time_display:.4f} s"
    )
    ax5.text(
        0.98,
        0.98,
        stats_text,
        transform=ax5.transAxes,
        verticalalignment="top",
        horizontalalignment="right",
        bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.8),
        fontsize=10,
        family="monospace",
    )
else:
    ax5.text(
        0.5,
        0.5,
        "Statistics not available\n(See Cell 11 for correlation analysis)",
        transform=ax5.transAxes,
        ha="center",
        va="center",
        fontsize=10,
    )

# Panel 6: Ellipse.Center.X comparison (bottom right) with dual y-axis
ax6 = fig.add_subplot(gs[3, 1])
ax6_twin = ax6.twinx()  # Create a second y-axis

if VideoData1_Has_Sleap and VideoData2_Has_Sleap:
    # Plot the individual traces
    ax6.plot(
        VideoData1["Seconds"],
        VideoData1["Ellipse.Center.X"],
        linewidth=0.5,
        c="#FF7F00",
        alpha=0.6,
        label="VideoData1 Ellipse.Center.X",
    )
    ax6.plot(
        VideoData2["Seconds"],
        VideoData2["Ellipse.Center.X"],
        linewidth=0.5,
        c="#9370DB",
        alpha=0.6,
        label="VideoData2 Ellipse.Center.X",
    )

    # Plot the difference on the right axis
    # Align the data to the same length and normalize for fair comparison
    min_length = min(len(VideoData1), len(VideoData2))

    # Normalize data (z-score) to account for different scales
    center_x1_aligned = VideoData1["Ellipse.Center.X"].iloc[:min_length]
    center_x2_aligned = VideoData2["Ellipse.Center.X"].iloc[:min_length]

    # Calculate mean and std for normalization
    mean1 = center_x1_aligned.mean()
    std1 = center_x1_aligned.std()
    mean2 = center_x2_aligned.mean()
    std2 = center_x2_aligned.std()

    # Normalize both datasets
    center_x1_norm = (center_x1_aligned - mean1) / std1
    center_x2_norm = (center_x2_aligned - mean2) / std2

    # Calculate difference of normalized data
    center_x_diff = center_x1_norm - center_x2_norm
    seconds_aligned = VideoData1["Seconds"].iloc[:min_length]
    ax6_twin.plot(
        seconds_aligned,
        center_x_diff,
        linewidth=0.5,
        c="green",
        alpha=0.6,
        label="Difference (normalized)",
    )

elif VideoData1_Has_Sleap:
    ax6.plot(
        VideoData1["Seconds"],
        VideoData1["Ellipse.Center.X"],
        linewidth=0.5,
        c="#FF7F00",
        alpha=0.6,
        label="VideoData1 Ellipse.Center.X",
    )
elif VideoData2_Has_Sleap:
    ax6.plot(
        VideoData2["Seconds"],
        VideoData2["Ellipse.Center.X"],
        linewidth=0.5,
        c="#9370DB",
        alpha=0.6,
        label="VideoData2 Ellipse.Center.X",
    )

ax6.set_xlabel("Time (s)")
ax6.set_ylabel("Center X (pixels)", color="black")
ax6.set_title("Ellipse.Center.X Comparison")
ax6.tick_params(axis="y", labelcolor="black")
if VideoData1_Has_Sleap and VideoData2_Has_Sleap:
    ax6_twin.set_ylabel("Normalized Difference (z-score)", color="green")
    ax6_twin.tick_params(axis="y", labelcolor="green")

# Combine legends from both axes
lines1, labels1 = ax6.get_legend_handles_labels()
if VideoData1_Has_Sleap and VideoData2_Has_Sleap:
    lines2, labels2 = ax6_twin.get_legend_handles_labels()
    ax6.legend(lines1 + lines2, labels1 + labels2, loc="upper left")
else:
    ax6.legend(loc="upper left")

ax6.grid(True, alpha=0.3)

# Add statistics text to Panel 6
if (
    pearson_r_center is not None
    and pearson_p_center is not None
    and peak_lag_time_center is not None
):
    stats_text = (
        f"Pearson r = {pearson_r_center:.4f}\n"
        f"Pearson p = {pearson_p_center:.4e}\n"
        f"Peak lag = {peak_lag_time_center:.4f} s"
    )
    ax6.text(
        0.98,
        0.98,
        stats_text,
        transform=ax6.transAxes,
        verticalalignment="top",
        horizontalalignment="right",
        bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.8),
        fontsize=10,
        family="monospace",
    )
else:
    ax6.text(
        0.5,
        0.5,
        "Statistics not available\n(both eyes required)",
        transform=ax6.transAxes,
        ha="center",
        va="center",
        fontsize=10,
    )

# Save as PDF (editable vector format)
save_path.mkdir(parents=True, exist_ok=True)
pdf_path = qc_debug_dir / "Eye_data_QC.pdf"
plt.savefig(pdf_path, dpi=300, bbox_inches="tight", format="pdf")
print(f"✅ QC figure saved as PDF (editable): {pdf_path}")

# Also save as 600 dpi PNG (high-resolution for printing)
png_path = qc_debug_dir / "Eye_data_QC.png"
plt.savefig(png_path, dpi=600, bbox_inches="tight", format="png")
print(f"✅ QC figure saved as PNG (600 dpi for printing): {png_path}")

plt.close(fig)

In [ ]:
# if needed, Create interactive time series plots using plotly for browser viewing
if plot_QC_timeseries:
    # Create subplots for the time series (3 rows now instead of 2)
    # Need to enable secondary_y for the third panel
    fig = make_subplots(
        rows=3,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.08,
        subplot_titles=(
            f"{get_eye_label('VideoData1')} - center.X Time Series",
            f"{get_eye_label('VideoData2')} - center.X Time Series",
            "Ellipse.Center.X Comparison with Difference",
        ),
        # Enable secondary_y for row 3
        specs=[[{}], [{}], [{"secondary_y": True}]],
    )

    # Panel 1: VideoData1 center coordinates - Time Series
    if VideoData1_Has_Sleap:
        fig.add_trace(
            go.Scatter(
                x=VideoData1_centered["Seconds"],
                y=VideoData1_centered["center.x"],
                mode="lines",
                name="center.x original",
                line=dict(color="blue", width=0.5),
                opacity=0.6,
            ),
            row=1,
            col=1,
        )

        fig.add_trace(
            go.Scatter(
                x=VideoData1["Seconds"],
                y=VideoData1["Ellipse.Center.X"],
                mode="lines",
                name="Ellipse Center.X",
                line=dict(color="red", width=0.5),
                opacity=0.6,
            ),
            row=1,
            col=1,
        )

    # Panel 2: VideoData2 center coordinates - Time Series
    if VideoData2_Has_Sleap:
        fig.add_trace(
            go.Scatter(
                x=VideoData2_centered["Seconds"],
                y=VideoData2_centered["center.x"],
                mode="lines",
                name="center.x original",
                line=dict(color="blue", width=0.5),
                opacity=0.6,
            ),
            row=2,
            col=1,
        )

        fig.add_trace(
            go.Scatter(
                x=VideoData2["Seconds"],
                y=VideoData2["Ellipse.Center.X"],
                mode="lines",
                name="Ellipse Center.X",
                line=dict(color="red", width=0.5),
                opacity=0.6,
            ),
            row=2,
            col=1,
        )

    # Panel 3: Ellipse.Center.X Comparison with difference
    if VideoData1_Has_Sleap and VideoData2_Has_Sleap:
        # Plot the individual traces
        fig.add_trace(
            go.Scatter(
                x=VideoData1["Seconds"],
                y=VideoData1["Ellipse.Center.X"],
                mode="lines",
                name="VideoData1 Ellipse.Center.X",
                line=dict(color="#FF7F00", width=0.5),  # Orange
                opacity=0.6,
            ),
            row=3,
            col=1,
        )

        fig.add_trace(
            go.Scatter(
                x=VideoData2["Seconds"],
                y=VideoData2["Ellipse.Center.X"],
                mode="lines",
                name="VideoData2 Ellipse.Center.X",
                line=dict(color="#9370DB", width=0.5),  # Purple
                opacity=0.6,
            ),
            row=3,
            col=1,
        )

        # Plot the difference on secondary y-axis
        # Align the data to the same length and normalize for fair comparison
        min_length = min(len(VideoData1), len(VideoData2))

        # Normalize data (z-score) to account for different scales
        center_x1_aligned = VideoData1["Ellipse.Center.X"].iloc[:min_length]
        center_x2_aligned = VideoData2["Ellipse.Center.X"].iloc[:min_length]

        # Calculate mean and std for normalization
        mean1 = center_x1_aligned.mean()
        std1 = center_x1_aligned.std()
        mean2 = center_x2_aligned.mean()
        std2 = center_x2_aligned.std()

        # Normalize both datasets
        center_x1_norm = (center_x1_aligned - mean1) / std1
        center_x2_norm = (center_x2_aligned - mean2) / std2

        # Calculate difference of normalized data
        center_x_diff = center_x1_norm - center_x2_norm
        seconds_aligned = VideoData1["Seconds"].iloc[:min_length]

        fig.add_trace(
            go.Scatter(
                x=seconds_aligned,
                y=center_x_diff,
                mode="lines",
                name="Difference (normalized)",
                line=dict(color="green", width=0.5),
                opacity=0.6,
            ),
            row=3,
            col=1,
            secondary_y=True,
        )

    elif VideoData1_Has_Sleap:
        fig.add_trace(
            go.Scatter(
                x=VideoData1["Seconds"],
                y=VideoData1["Ellipse.Center.X"],
                mode="lines",
                name="VideoData1 Ellipse.Center.X",
                line=dict(color="#FF7F00", width=0.5),
                opacity=0.6,
            ),
            row=3,
            col=1,
        )
    elif VideoData2_Has_Sleap:
        fig.add_trace(
            go.Scatter(
                x=VideoData2["Seconds"],
                y=VideoData2["Ellipse.Center.X"],
                mode="lines",
                name="VideoData2 Ellipse.Center.X",
                line=dict(color="#9370DB", width=0.5),
                opacity=0.6,
            ),
            row=3,
            col=1,
        )

    # Update layout
    fig.update_layout(
        height=1200,  # Increased height for 3 panels
        title_text=f"{data_path} - Eye Tracking Time Series QC",
        showlegend=True,
        hovermode="x unified",
    )

    # Update axes
    fig.update_xaxes(title_text="Time (s)", row=3, col=1)
    fig.update_yaxes(title_text="Position (pixels)", row=1, col=1)
    fig.update_yaxes(title_text="Position (pixels)", row=2, col=1)
    fig.update_yaxes(title_text="Center X (pixels)", row=3, col=1)

    # Update secondary y-axis for difference plot
    if VideoData1_Has_Sleap and VideoData2_Has_Sleap:
        fig.update_yaxes(
            title_text="Normalized Difference (z-score)", row=3, col=1, secondary_y=True
        )

    # Show in browser
    fig.show(renderer="browser")

    # Also save as HTML
    save_path.mkdir(parents=True, exist_ok=True)
    html_path = qc_debug_dir / "Eye_data_QC_time_series.html"
    fig.write_html(html_path)
    print(f"✅ Interactive time series plot saved to: {html_path}")



In [ ]:
# SAVE PROCESSED VIDEO DATA (PRE-SACCADE NOTEBOOK OUTPUTS)
##########################################################################
# Save cleaned per-frame data and resampled versions for downstream notebooks.
# The saccade notebook (1_3) should load these files directly.

save_path.mkdir(parents=True, exist_ok=True)
downsampled_output_dir = save_path / "downsampled_data"
downsampled_output_dir.mkdir(parents=True, exist_ok=True)

video_output_paths = {}

for video_key, has_sleap, video_df, eye_tag in [
    ("VideoData1", VideoData1_Has_Sleap, VideoData1 if "VideoData1" in globals() else None, "eye1"),
    ("VideoData2", VideoData2_Has_Sleap, VideoData2 if "VideoData2" in globals() else None, "eye2"),
]:
    if not has_sleap or video_df is None:
        continue

    # Prefer all detected blink segments (including short ones), fallback to filtered list.
    if video_key == "VideoData1":
        blink_segments_for_export = (
            all_blink_segments_v1
            if "all_blink_segments_v1" in globals()
            else (blink_segments_v1 if "blink_segments_v1" in globals() else [])
        )
    else:
        blink_segments_for_export = (
            all_blink_segments_v2
            if "all_blink_segments_v2" in globals()
            else (blink_segments_v2 if "blink_segments_v2" in globals() else [])
        )

    export_df = add_blink_flag_column(video_df, blink_segments_for_export)

    # Save full-rate processed data.
    export_df_for_save = (
        export_df.rename(columns={"Ellipse.Diameter.Filt": "Pupil.Diameter"})
        if "Ellipse.Diameter.Filt" in export_df.columns
        else export_df
    )
    # Save resampled version for multimodal alignment workflows.
    drop_for_resample = [col for col in RESAMPLED_DROP_COLUMNS if col in export_df.columns]
    export_df_resampled_input = export_df.drop(columns=drop_for_resample)
    export_df_resampled = resample_video_dataframe(export_df_resampled_input, eye_tag)
    if "blink_flag" in export_df_resampled.columns:
        export_df_resampled["blink_flag"] = (
            pd.to_numeric(export_df_resampled["blink_flag"], errors="coerce")
            .fillna(0)
            .ge(0.5)
            .astype("Int64")
        )
    if "Ellipse.Diameter.Filt" in export_df_resampled.columns:
        export_df_resampled = export_df_resampled.rename(
            columns={"Ellipse.Diameter.Filt": "Pupil.Diameter"}
        )

    export_df_resampled_indexed = set_aeon_index(export_df_resampled)
    resampled_parquet_path = downsampled_output_dir / f"{video_key}_resampled.parquet"
    export_df_resampled_indexed.to_parquet(
        resampled_parquet_path, engine="pyarrow", compression="snappy"
    )
    print(f"✅ Saved {video_key} resampled parquet to {resampled_parquet_path}")

    video_output_paths[video_key] = {
        "resampled_parquet": str(resampled_parquet_path),
    }

# Save run metadata for the downstream saccade notebook.
metadata = {
    "created_at": datetime.now().isoformat(),
    "data_path": str(data_path),
    "save_path": str(save_path),
    "common_resampled_rate_hz": COMMON_RESAMPLED_RATE,
    "video1_eye": video1_eye,
    "video2_eye": video2_eye,
    "max_low_confidence_interpolation_fraction": max_low_confidence_interpolation_fraction,
    "fps": {
        "VideoData1": float(FPS_1) if "FPS_1" in globals() and FPS_1 is not None else None,
        "VideoData2": float(FPS_2) if "FPS_2" in globals() and FPS_2 is not None else None,
    },
    "outputs": video_output_paths,
    "eye_with_least_low_confidence": (
        eye_with_least_low_confidence
        if "eye_with_least_low_confidence" in globals()
        else None
    ),
    "low_confidence_counts": (
        low_confidence_counts
        if "low_confidence_counts" in globals()
        else {"VideoData1": None, "VideoData2": None}
    ),
    "low_confidence_percentages": (
        low_confidence_percentages
        if "low_confidence_percentages" in globals()
        else {"VideoData1": None, "VideoData2": None}
    ),
    "excluded_videodata_due_to_low_confidence": (
        excluded_videodata_due_to_low_confidence
        if "excluded_videodata_due_to_low_confidence" in globals()
        else []
    ),
}

metadata_path = downsampled_output_dir / "saccade_input_metadata.json"
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)

print(f"✅ Saved metadata to {metadata_path}")

